# <center>编译式RAG第三节课：用 GBrain 搭建你的第二大脑</center>

&emsp;&emsp;前两节课，我们把 GBrain 这台机器拆开看了个透。第一节我们讲清楚了它的世界观——编译式 RAG 的范式、`init / import / extract` 三个核心动作、self-wiring 自动建图、以及它对自己能力边界的诚实标定；第二节我们钻进引擎舱，看了 Storage 层（PGLite 与 Postgres 两个引擎、一套 BrainEngine 契约）、Retrieval 层（HNSW 向量 + BM25 关键词 + RRF 融合）、从 `search` 到 `think` 的综合层跨越，以及围绕它的技能生态和 benchmark 体系。

&emsp;&emsp;但拆解归拆解——我们一直没有真正动手，从零搭一个属于自己的大脑。这节课，我们要把前两节的所有零件组装成一条完整的端到端链路：**从设计大脑形状，到建一个带向量检索的库，导入你自己的多源知识，用 `think` 做跨源综合，最后通过 MCP 把这个大脑接进 Claude Code，让它成为一个跨会话不失忆的 Agent 长记忆**。这不是又一遍原理课，而是一次完整的落地实操——每一步你都能在自己的机器上跑出来、验收掉。

> 📌 **目标受众与前置要求**：本节面向已经完成第一节《编译式 RAG 原理详解》和第二节《GBrain 核心功能详解》的学员。技术上你需要：装好 `gbrain`（v0.42.x 系列）、有一个能用的 LLM API key（本节用 DeepSeek 做 `think` 的合成模型）、有一个 embedding provider 的 key（DashScope 或 OpenAI 兼容代理任一）。你**不需要** 重新学三层架构原理、self-wiring 建图机制、RRF 融合内部算法——这些第一、二节已经讲透，本节只在用到时做 30 秒回顾。

> 📌 **学完本节你将带走 7 件产物**：① 一套为自己领域设计的大脑 schema（type + 关系动词）；② 两个真实建成的、带向量检索的 GBrain 库——7 页的 `brain-l3`（看清建库每一步机制）和 154 页的 `brain-vc`（后续 `think`、接 Agent 都在它上面做），都隔离在 `./output/`，不污染你现有的大脑；③ 一条可复用的多源知识导入流程；④ 一次 `think` 会前综合的真实输出（带来源引用 + 空白分析）；⑤ 一份写进 `CLAUDE.md` 的 brain-first 行为契约；⑥ 一次跨会话长记忆的受控验证经验；⑦ 一张 GBrain vs Mem0/Zep 的选型判断框架。

> **学完不能做（诚实划界）**：本节不会让你掌握团队大脑的 Postgres 完整迁移 + 多 source 安全隔离实操、混合架构的两套 RAG 系统搭建、cron 自动化管线的生产部署——这些进阶路径我们在第 7 章速览（团队 brain 的"内容面"会真跑一次演示，但隔离配置、混合架构、cron 这些需要额外环境的部分只展示不真跑），每条都有对应的部署指南供你后续自学深入。本节也不承诺 GBrain 是"建好就一劳永逸"的——KB 会漂移、会自我投毒，这是第 6 章要正面处理的。

> 📅 **时效性说明**：本节全部命令基于 `gbrain v0.42.x 系列`（实测环境 0.42.44.0，2026-06-21 验证）。GBrain 的小版本迭代频繁，偶有 breaking 变更——所以我们锁的是 "v0.42.x 系列" 这个口径，不钉死某个小版本。本课所有命令和文本输出都基于真实实测；其中第 5 章跨会话演示的 6 张截图正在按当前 `brain-vc` 版本重录更新，其余截图均为真实跑出来的结果。你也可以在自己的隔离库（`brain-l3` 练机制、`brain-vc` 做应用）上复跑每一个真跑步骤。

---

## <center>第 1 章：目标与全景</center>

&emsp;&emsp;动手搭一套东西，最容易陷进去的坑就是：埋头敲了两小时命令，到头来还是不知道自己在拼一张什么样的图。所以这一章我们反过来——先把终点摆出来，让你亲眼看见"搭完之后到底能用它做什么"，再倒推回去看每一步是为了什么。先看终态，再回看实现它所需的步骤。

### 1.1 第二大脑解决什么

&emsp;&emsp;"第二大脑"这个词被用滥了，很多人以为就是个高级笔记软件。我们这里说的第二大脑，要解决的是两个非常具体的痛点，而且这两个痛点是连在一起的。

&emsp;&emsp;第一个痛点：**你的知识散落在各处**。会议纪要在飞书、决策记录在 Notion、人物关系记在脑子里、项目背景埋在一堆 Markdown 文件里。当你要回答"上次跟 Alex 聊增长策略，最后定了什么"时，你得自己在四五个地方翻。第二个痛点更隐蔽：**你的 AI 助手每开一个新会话就从零开始**。你昨天花半小时跟 Claude Code 讨论清楚的架构决策，今天新开一个窗口，它完全不记得——你又得把背景重讲一遍。

&emsp;&emsp;这两条线，GBrain 用同一个机制统一解决：把你的知识**编译** 成一个带图谱、带向量检索的大脑，既能让你自己跨源查询，又能通过 MCP 接进 Agent，让 Agent 把它当长记忆来读写。知识沉淀和跨会话记忆，本质是同一个大脑的两个出口。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171243176.png" width=50%></div>

### 1.2 终点预览：跨会话记忆命中

&emsp;&emsp;空讲两个痛点没有说服力，我们直接看终态。下面这张图是真实跑出来的结果：一个 Claude Code 会话（我们叫它会话 A）把一条决策写进了大脑，然后**完全关闭**；接着新开一个零对话历史的会话 B，问它"现在我们项目的当前状态是什么"——会话 B 在没有任何上下文喂给它的情况下，主动查了大脑、命中了会话 A 写入的事实、并引用了来源页。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171247244.png" width=50%></div>

&emsp;&emsp;请仔细看这张截图里的几个关键点：会话 B 显示 "Called gbrain 3 times"——它自己决定去查了大脑三次；命中后给出的答案明确标了来源页 `facts/brain-first-protocol-adopted-2026-06`；而且它复述出了完整的三条 brain-first 契约（Search-First 答前先查 / Write-Back 答后写回 / Cite 引用来源）。这就是我们这节课要搭的终态——一个新会话，零历史，却能记得上一个会话教给大脑的东西。

&emsp;&emsp;另一半终态，是你对自己的大脑跑一次 `think` 会前综合时拿到的东西。我们第 4.2 节会真跑这个演示、给出完整问题和输出，这里先给你看它的节选长什么样：

```
# Garry Tan 最近在投资组合上有哪些关键决策和待办行动项？有没有盲区？
#（下面是 §4.2 完整会前综合输出的节选）

## 关键决策
1. OpenAI 条款修改：估值上限 4.88 亿美元、18% pro-rata [email-2026-03-20-0042-openai]
2. Scale AI 条款修改：估值上限 7.34 亿美元 [email-2026-03-15-0037-scale-ai]

## 待办行动项
- 跟进 OpenAI×Stripe 合作对 Stripe 持仓的影响 [slack-weekly-sync-2026-04-13-0026]

## 潜在盲区
- OpenAI 流失率 3.4% 高于行业均值 2.1%，却没人深究
- Garry Tan 同时领多项集成规划 + 多家条款谈判，个人带宽吃紧

## Gaps
- 各合作项目（Rippling / Stripe）的时间表和里程碑
- 对高流失率（OpenAI 3.4%）的深入分析和应对

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 29
```

&emsp;&emsp;这是问"Garry Tan 最近在投资组合上有哪些关键决策和待办？有没有盲区"得到的答案节选。它不是给你一堆页面让你自己读，而是直接成文：分了"关键决策""待办行动项""潜在盲区""Gaps"几段，每条都带来源页引用。底部那行 `Citations: 29` 是它综合引用的来源处数（完整问题和输出在 4.2 节「会前综合演示」）。这两个终点——跨会话命中 + 会前综合——就是本节要带你亲手做出来的东西。接下来的每一章，都是为了让你能做出这两件事。

### 1.3 与第一节的不同：全程带 embedding

&emsp;&emsp;如果你还记得第一节的实操，我们当时建库、导入、抽链接，命令后面几乎都跟着一个 `--no-embed`。那是有意为之——第一节我们聚焦的是 self-wiring 图谱那一层，图谱靠的是 wikilink 解析，不需要向量。所以第一节的大脑，只有图谱、没有向量检索。

&emsp;&emsp;这一节，我们要把跳过的那一半补上。**全程带 embedding**，意味着你导入的每一页都会被切块、向量化，存进 pgvector。这样才能用上第二节讲的混合检索（HNSW 向量 + BM25 + RRF），`think` 也才能真正"读懂"语义相近但用词不同的页面。下面这张表把两节的形态对照清楚：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>第一节 vs 本节：大脑能力形态对照</font></p>
<div class="center">

| 维度 | 第一节（`--no-embed`） | 本节（带 embedding） |
|------|----------------------|---------------------|
| 建库命令 | `gbrain init`（不配 embedding） | `gbrain init --embedding-model <provider>:<model>` |
| 导入 | `gbrain import <dir> --no-embed` | `gbrain import <dir>`（默认嵌入） |
| 大脑里有什么 | wiki 页 + self-wiring 图谱 | wiki 页 + 图谱 + **向量索引** |
| 能做的检索 | 关键词 + 图谱遍历 | 关键词 + 图谱 + **语义向量检索** |
| `think` 综合 | 受限（无向量召回） | 完整（向量召回多页再综合） |
| 对应章节 | 第一节三、四章 | 本节第 3、4 章 |

</div>

&emsp;&emsp;这张表也解释了为什么本节的顺序是"先建带 embedding 的库（第 3 章），再做 `think` 综合（第 4 章）"——没有向量，`think` 的召回会瘸腿。这是一条硬依赖，我们后面会严格按它走。

### 1.4 本节完整链路预览

&emsp;&emsp;为了让你心里有张地图，我们把整节的链路画出来。这条链路有 5 个里程碑，每一个都对应后面的一章或几节，每一个都做到"可验收"才往下走——这是我们这节课的纪律：即学即落，不攒到最后。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171241443.png" width=50%></div>

&emsp;&emsp;你今天会依次经历这 5 个里程碑：**第一个**，在第 2 章设计大脑的形状——决定你要存哪些类型的知识、它们之间有什么关系；**第二个**，在第 3 章建一个带 embedding 的库，把多源知识导进去编译；**第三个**，还在第 3 章，用 `stats / list / graph` 三件套验收"大脑真的长出来了"；**第四个**，在第 4 章用 `think` 做一次会前综合，亲手拿到带引用 + 空白分析的答案；**第五个**，在第 5 章把大脑接进 Claude Code，验证跨会话长记忆。走完这五步，本节开头那两张终态截图，就是你自己能做出来的东西了。

### 1.5 本节要回收的三个痛点

&emsp;&emsp;落地之前，我们先把要解决的痛点钉在墙上。下面三个痛点会在后面对应的章节被逐一回收验证——这是我们对你的承诺，也是你检验自己有没有真学会的标尺。

> **【痛点一 · 知识散落各处，AI 每次从零开始】**　你的知识分散在不同工具里，AI 助手帮不上忙，因为它看不到全貌。**回收点：第 3 章 3.6 / 3.7 节**——当你 import 进多源知识后跑 `stats`，会看到一个真实长出来的大脑；3.7 节我们会真的导入一份 154 页的多源语料，self-wiring 在它上面抽出 **1172 条边**。

> **【痛点二 · 搜索只返回页面，还要自己读】**　传统检索给你一串链接，你得自己点开读完才能用。**回收点：第 4 章 4.1 节**——`search` vs `think` 对照演示，`think` 直接给你成文答案 + 空白分析，不用自己读。

> **【痛点三 · Agent 每开新会话就失忆】**　这是最核心的痛点，也是本节的主战场。**回收点：第 5 章 5.4-5.5 节**——写一份 brain-first 契约进 `CLAUDE.md`，然后跨会话验证，让新会话记得上个会话的事实。

&emsp;&emsp;痛点钉好了，地图也有了。我们从第一个里程碑出发——在动手建库之前，先想清楚一件事：你的大脑应该是什么形状？这就是第 2 章要回答的问题。

---

## <center>第 2 章：设计大脑形状（schema）</center>

&emsp;&emsp;很多人拿到 GBrain 第一反应是直接 `init` 建库、`import` 导入，然后发现大脑里的页面类型一团乱——人物、会议、决策全堆在一起，图谱连不起来。问题出在跳过了第一步：**先想清楚你的大脑里要装哪些类型的知识，它们之间有什么关系**。这一步叫设计 schema，是建库之前的地基。这一章不长，但它决定了后面三章建出来的大脑能不能用。

&emsp;&emsp;好消息是，你不需要从一张白纸开始写 schema。GBrain 自带了一个相当完整的基础包，覆盖了大多数个人知识场景。我们这一章先看怎么用现成的包，再看怎么在它基础上扩展出你自己领域的类型。

### 2.0 环境准备

&emsp;&emsp;在跑任何命令之前，我们先把实验环境隔离好。这一节课主要在一个独立的大脑 `./output/brain-l3` 上操作——它建在课件目录下，和你真正在用的大脑分开，随便折腾、删库重建都不影响别的库。隔离的关键在**建库这一步**：第 3 章我们会用 `gbrain init --path ./output/brain-l3` 把大脑创建在这个目录，而 `init` 同时会把它设成"**活动库**"。之后的 `import` / `stats` / `think` 等命令都自动操作这个活动库——**这里先记住一个关键事实：只有 `init` 接受 `--path`，其它读命令（think/search/stats/list/graph 等）都不认 `--path`，它们永远操作最近一次 `init` 选定的那个活动库**（到第 4 章需要换库时我们会再实操一次）。

&emsp;&emsp;下面这个 cell 是本节所有命令的共享前置——它做一件事：用 Python 的 `load_dotenv` 把 API key 加载进 `os.environ`。这一点很关键：Jupyter 里每个 `!` shell 命令是独立子进程，但它们会**继承当前 Python 进程的 `os.environ`**。所以只要这个 Python cell 跑过一次，后面所有 `!gbrain` 命令就都能拿到 key——这比用 bash `source` 可靠得多（bash 的环境变量不跨 cell 持久）。

In [ ]:
# 用 Python load_dotenv 加载 key，后续所有 !gbrain 命令继承 os.environ 里的 key
from dotenv import load_dotenv
import os

# DASHSCOPE 做 embedding（向量化），DEEPSEEK 做 think（综合）
load_dotenv(os.path.expanduser("~/.claude/.env"))

# gbrain 跑在 Bun 运行时上，Bun 的 TLS 验证偶发抽风，会报 "unknown certificate verification error"
# （curl 直测 DashScope/DeepSeek 端点证书其实有效，是 Bun 自己的 TLS bug，会命中 embedding/think 的 API 调用）。
# 设下面这个变量能"降低"它出现的频率，但**不保证根除**（它是间歇性的）。真正可靠的兜底是"重跑"：
# import 撞到会跳过那一页、重跑 import 会自动补上；think 撞到，把那条 think 重跑一遍即可（都是瞬时错误）。
# [警告] 它会关闭本进程的 TLS 证书校验，仅适合本机 + 可信端点（官方 DashScope/DeepSeek）；生产环境请按规范处理 TLS。
os.environ["NODE_TLS_REJECT_UNAUTHORIZED"] = "0"

# 隔离大脑路径：建在课件当前目录的 ./output 下，跑完命令就能直接看到产出（不碰 ~/.gbrain）
BRAIN_PATH = "./output/brain-l3"

# 确认 key 已就位（只打印状态，不打印 key 值）
print("DASHSCOPE:", "已就位" if os.getenv("DASHSCOPE_API_KEY") else "缺失")
print("DEEPSEEK: ", "已就位" if os.getenv("DEEPSEEK_API_KEY") else "缺失")

&emsp;&emsp;看到两个 key 都"已就位"，说明环境准备好了。接着我们做最后一步确认——`gbrain` 命令本身是否装好、版本对不对：

In [ ]:
# 确认 gbrain 可用、版本在 v0.42.x 系列
!gbrain --version

> **【踩坑预警 · 大脑数据必须落 APFS】**　本节把大脑建在课件目录下的 `./output`，**前提是课件放在内置盘（macOS 内置盘是 APFS）**。如果你在 Mac 外置盘（通常是 exFAT）上建大脑，PGLite 的 WASM mmap 会因权限冲突起不来。**正确做法**：把课件（连同 `./output`）放在内置 APFS 盘；如果课件不得不放外置 exFAT 盘，就把 `init` 的 `--path` 改指向内置 APFS 路径（如 `~/brain-l3`），让 Markdown 源放外置盘、大脑落本地 APFS。**排查方法**：建库报 mmap/permission 相关错误时，先确认大脑目录落在哪个文件系统，`df <目录>` 看是不是 exFAT。

&emsp;&emsp;环境准备好了——key 由上面的 Python cell 加载进 `os.environ`，大脑会由第 3 章第一条 `gbrain init --path ./output/brain-l3` 建好并设为活动库。后面每一章的命令都默认这个 Python cell 已经执行过；除了 `init`，其它命令都不写 `--path`，直接操作当前活动库。

### 2.1 梳理你的知识形态

&emsp;&emsp;设计 schema 不是技术活，是想清楚"我的知识长什么样"。在动任何命令之前，我们先做一个思维练习。回想一下你日常产生的知识，它们大致能归成几种形态：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>个人知识的五种常见形态</font></p>
<div class="center">

| 知识形态 | 典型 type | 例子 | 关心什么 |
|---------|----------|------|---------|
| 人 | `person` | 同事、客户、合作伙伴 | 这个人是谁、负责什么、参与了哪些事 |
| 组织 | `company` | 公司、团队、项目方 | 这家公司在做什么、谁在里面 |
| 事件 | `meeting` | 会议、通话、评审 | 谁参加了、讨论了什么、定了什么 |
| 决策 | `decision` | 拍板的事、待办 | 决定了什么、谁牵头、什么状态 |
| 散记 | `note` | 想法、背景、原则 | 记下来防止忘，可能还没成型 |

</div>

&emsp;&emsp;这五种形态几乎覆盖了个人知识管理的主体。当你想清楚"我主要在记这五种东西"，schema 设计就成功了一半——剩下的是给它们之间连上关系（谁参加了哪个会议、哪个决策来自哪次讨论），这就是 wikilink 干的活，第一节四章我们已经讲过。

### 2.2 schema active：查看当前 schema

&emsp;&emsp;想清楚知识形态之后，好消息是你不用自己从零定义这些 type——GBrain 已经内置了一个非常完整的基础 schema 包 `gbrain-base-v2`，把常见类型都帮你定义好了。我们直接看这个现成包里有什么类型就够了。两个命令：`schema active` 看当前激活的是哪个包，`schema list` 看有哪些包可用。**注意：这几个 `schema` 命令读的是 gbrain 全局配置里的 schema 包（`Source: home-config`），不依赖任何具体大脑库——所以现在还没到第 3 章建库，照样能看。**

In [ ]:
# 看当前激活的 schema 包：包名、版本、类型数、关系动词数（读全局 schema 配置，不带 --path）
!gbrain schema active

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171256943.png" width=72%></div>

&emsp;&emsp;这个命令的输出告诉我们几件关键的事：当前激活的包是 `gbrain-base-v2 v1.0.0`，它定义了 **15 个页面类型（Page types）、14 个关系动词（Link verbs）**，还有几种 takes 类型（fact / take / bet / hunch）。这 15 个类型就是我们刚才那张"五种知识形态"表的超集——它远不止五种，但你日常用到的就是其中那几个核心类型。

In [ ]:
# 看所有可用的 schema 包（bundled 自带的 + 用户安装的）
!gbrain schema list

&emsp;&emsp;`schema list` 会列出 bundled（自带）的包——你会看到 `gbrain-base` 和 `gbrain-recommended`。如果你之前没装过自定义包，"user-installed"那一栏会是空的。这就是 GBrain 的设计哲学：**给你一套足够好的默认 schema，让你开箱即用，而不是逼你从零设计**。对绝大多数个人大脑，直接用 `gbrain-base-v2` 就够了。

### 2.3 gbrain-base-v2 的 15 个类型

&emsp;&emsp;光说"15 个类型"太抽象，我们用 `schema graph` 把它们和各自的分类、别名都看清楚。这个命令把整个 schema 包的类型结构画出来：

In [ ]:
# 看 schema 包的类型/关系图谱：每个类型属于哪个大类、有哪些别名
!gbrain schema graph

&emsp;&emsp;真跑出来你会看到 15 个类型按 5 个大类（entity 实体 / media 媒体 / temporal 时序 / concept 概念 / annotation 标注）组织。这个分类很有讲究——它解释了为什么有些类型天然有时间属性（gbrain-base-v2 里 temporal 类的规范类型是 `deal`、`email`、`slack`、`social-digest`），有些是稳定实体（`person`、`company` 属于 entity）。这里要诚实说明一点：gbrain 也接受**自定义类型**——本节语料我们会用到的 `meeting`（会议/事件）就不在这 15 个规范类型里，但 gbrain 会按你写的 `type` 原样存下来，语义上它和 temporal 那几个一样是带时间的事件页。下面这张表把最常用的几个挑出来：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>gbrain-base-v2 常用类型（节选自 15 类型）</font></p>
<div class="center">

| type | 大类 | 别名（aliases） | 用来记 |
|------|------|----------------|--------|
| `person` | entity | people, contact, founder, partner | 人物 |
| `company` | entity | org, startup, business, product | 组织 |
| `email` | temporal | email-thread | 邮件往来 |
| `deal` | temporal | investment, term-sheet | 交易/投资 |
| `meeting`（自定义类型，非规范 15 类） | 自定义·语义近 temporal | —— | 会议/事件，本节语料用到 |
| `note` | concept | memo, insight, principle, framework | 散记/原则 |
| `project` | concept | initiative, workstream | 项目/工作流 |
| `concept` | concept | definition | 概念定义 |
| `analysis` | media | market-analysis, research | 分析报告 |

</div>

&emsp;&emsp;注意"别名"这一栏的价值：当你导入一个 frontmatter 写 `type: founder` 的页面时，GBrain 会自动把它归到 `person` 类型——因为 `founder` 是 `person` 的别名。这让你导入别人的语料时不用逐个改 type，包已经帮你做了归一。

> **【常见误区 · wikilink 目录前缀不能漏】**　这里 30 秒回顾第一节四章讲过的一条铁律：写 wikilink 时，目标 slug 要带目录前缀。比如指向 `companies/harmony-ai.md` 这个页，wikilink 要写全 `[[companies/harmony-ai]]`，而不是只写 `[[harmony-ai]]`。**后果**：前缀对不上，抽图谱边时这条边就连不起来，图谱会断连。**排查方法**：抽完边后 `gbrain stats` 看 Links 数，如果远小于你预期的连接数，多半是 wikilink 没匹配上目标 slug。第 3 章我们会真实把这套带前缀的 wikilink 抽成 12 条边。

### 2.4 动手：扩展一个自定义 type

&emsp;&emsp;到这里你已经知道怎么用现成的 `gbrain-base-v2` 包了。最后我们做一个小练习——这是本章唯一需要你动脑的地方。`gbrain-base-v2` 很全，但它不可能覆盖你所有的专业领域。假设你做的是客户关系管理，你可能需要一个标准包里没有的类型，比如 `ticket`（工单）或 `contract`（合同）。

&emsp;&emsp;我们不需要你现在就写出完整的包定义（那是更进阶的内容），只需要你在脑子里走一遍设计的思路。问自己三个问题，把答案写下来：

> **【动手练习 · 设计你领域的 schema 草图】**
> 1. **我有哪些标准包覆盖不到的知识类型？**（例：客服领域的 `ticket`、法务领域的 `contract`、研究领域的 `experiment`）
> 2. **这个新类型和现有类型有什么关系？**（例：`ticket` 由某个 `person` 提出、关联某个 `company`）
> 3. **这个关系用哪个动词描述？**（例：`raised_by`、`belongs_to`——这就是 link verb）

&emsp;&emsp;举个填好的例子让你对照。如果我做的是客户支持，我的草图会是这样：新增类型 `ticket`（工单），它通过 `raised_by` 关系连到一个 `person`，通过 `about` 关系连到一个 `company`。这三件事一想清楚，你就从"我有什么知识"推到了"我需要什么 type 和关系"——这正是 schema 设计的全部。真要把它落成可安装的包，`gbrain schema init <name>` 能帮你 scaffold 一个继承 `gbrain-base` 的新包骨架，细节可查 `gbrain schema --help`。

&emsp;&emsp;本章小结：我们确认了大脑的形状——直接用 `gbrain-base-v2` 这个 15 类型的现成包，用 `schema active / list / graph` 看清它有哪些类型，并学会了为自己领域扩展类型的思路。形状定好了，下一章我们就真正动手建库、把知识导进去。

---

## <center>第 3 章：建库与导入（编译）</center>

&emsp;&emsp;这是本节第一个核心落地章——走完它，你手里就有一个真实的、带向量检索的大脑了。我们要做四件事：建一个带 embedding 的库、准备多源语料、把它们 import 进去编译、然后用三件套验收"大脑真的长出来了"。这一章的每个命令都会真跑，你跟着敲就能在自己的 `./output/brain-l3` 上得到一样的结果。

&emsp;&emsp;在开始之前，先把第 1 章那条硬依赖再钉一遍：**这一章全程带 embedding**。这是和第一节最大的区别。没有 embedding，第 4 章的 `think` 就召回不全。所以建库这一步，配好 embedding provider 是重中之重。

### 3.1 init 带 embedding

&emsp;&emsp;第一节我们建库时跳过了 embedding。这一章我们要显式启用它。命令本身不复杂，关键在 `--embedding-model` 这个参数——它告诉 GBrain 用哪个 provider 的哪个模型来做向量化。

&emsp;&emsp;这里有个**很坑的真实行为** 必须先讲清，否则你很可能建出一个没有向量能力的库还不自知：`gbrain init` **不显式传 `--embedding-model` 时默认走 `--no-embedding`（deferred setup）**，而且会在**全局** `~/.gbrain/config.json` 里写下一个 `embedding_disabled: true` 开关。这个开关很黏——它会**压倒你之后所有 init 的 `--embedding-model`**，哪怕带上 `--force` 也会被压成 no-embedding（连官方报错提示的 `gbrain init --force` 都救不了），而 `gbrain config set / unset` 也清不掉它（这两个命令只动 active-brain 配置、不动全局）。所以本节的做法是：**先用一个 Python cell 确保全局没有这个遗留开关，再从一开始就显式带 `--embedding-model` + `--force` 建库。**

In [ ]:
# 防御：清掉全局 ~/.gbrain/config.json 里可能遗留的 embedding_disabled 开关
# （之前若跑过不带 --embedding-model 的 init 会留下它，污染本次 embedding 建库；
#   gbrain config set/unset 清不掉，必须直接删这个 key）
import json, os
_cfg = os.path.expanduser("~/.gbrain/config.json")
if os.path.exists(_cfg):
    _c = json.load(open(_cfg))
    if _c.pop("embedding_disabled", None) is not None:
        json.dump(_c, open(_cfg, "w"), ensure_ascii=False, indent=2)
        print("已清除遗留的 embedding_disabled 开关")
    else:
        print("全局配置干净，无遗留开关")
else:
    print("尚无全局配置，干净")

In [67]:
# init 带 embedding：显式指定 provider:model + --force（强制按 embedding 重新 size schema）
# 主口径用 dashscope:text-embedding-v3（与第二节一致，1024 维）
# 若 DashScope key 不可用，可换 openai:text-embedding-3-small（1536 维，走 OpenAI 兼容代理）
!gbrain init --embedding-model dashscope:text-embedding-v3 --force --path ./output/brain-l3

Setting up local brain with PGLite (no server needed)...
  Embedding: dashscope:text-embedding-v3 (1024d)
  Expansion: openai:gpt-5.2
  Schema version 1 → 117 (112 migration(s) pending)
  [2] slugify_existing_pages...
  [2] ✓ slugify_existing_pages
  [3] unique_chunk_index...
  [3] ✓ unique_chunk_index
  [4] access_tokens_and_mcp_log...
  [4] ✓ access_tokens_and_mcp_log
  [5] minion_jobs_table...
  [5] ✓ minion_jobs_table
  [6] agent_orchestration_primitives...
  [6] ✓ agent_orchestration_primitives
  [7] agent_parity_layer...
  [7] ✓ agent_parity_layer
  [8] multi_type_links_constraint...
  [8] ✓ multi_type_links_constraint
  [9] timeline_dedup_index...
  [9] ✓ timeline_dedup_index
  [10] drop_timeline_search_trigger...
  [10] ✓ drop_timeline_search_trigger
  [11] links_provenance_columns...
  [11] ✓ links_provenance_columns
  [12] budget_ledger...
  [12] ✓ budget_ledger
  [13] minion_quiet_hours_stagger...
  [13] ✓ minion_quiet_hours_stagger
  [14] pages_updated_at_index...
  [14] ✓ 

&emsp;&emsp;这条命令做了一连串事情：建立 PGLite 本地库、跑完所有 schema migration、把 `embedding_model` 配成你指定的 provider，最后会问你要不要装 skill 包（这一步可以先跳过，回答 No）。建完之后，我们用 `config show` 确认 embedding 字段真的配上了：

In [68]:
# 确认 embedding 配置：embedding_model 和维度
!gbrain config show | grep -iE 'embedding_model|dimensions|expansion'

  expansion_model: openai:gpt-5.2
  embedding_model: dashscope:text-embedding-v3
  embedding_dimensions: 1024


&emsp;&emsp;你应该看到 `embedding_model` 那一行显示了你指定的 provider:model，`embedding_dimensions` 显示对应维度（DashScope v3 是 1024，OpenAI 3-small 是 1536）。这一行非常关键——它是"embedding 已经真正启用"的凭证。

> **【踩坑预警 · import 仍报 no-embedding 时怎么自救】**　如果你 `config show` 看到 `embedding_model` 明明有值、`import` 却还报 `This brain was initialized with --no-embedding`，**别只信 `config show` 的字段**——真正卡你的是全局 `~/.gbrain/config.json` 里那个 `embedding_disabled: true`。它由"曾经跑过不带 `--embedding-model` 的 init"留下，黏在全局、压倒后续所有 init 的 `--embedding-model`，而且 `gbrain config set / unset` 清不掉（只动 active-brain、不动全局）、`gbrain init --force` 也压不过它。**正确做法**：就是上面那个 Python cell 干的事——直接从 `~/.gbrain/config.json` 删掉 `embedding_disabled` 这个 key，再 `gbrain init --force --embedding-model <provider>:<model>` 重建，import 就正常了。

In [69]:
# Tier 1 验证：stats 确认 embedding 已就绪（此刻库还是空的，重点看命令能跑通、配置已生效）
!gbrain stats

Pages:     0
Chunks:    0
Embedded:  0
Links:     0
Tags:      0
Timeline:  0

By type:


&emsp;&emsp;现在库是空的（Pages 全是 0），但这一步的目的不是看数字，而是确认"带 embedding 的空库已经建好、`stats` 命令能正常跑"。地基打好了，下一步往里装知识。

### 3.2 语料准备：多源目录组织

&emsp;&emsp;import 之前，我们先把要导入的 Markdown 文件按类型组织好。这一步看似琐碎，但它直接决定了 schema 映射对不对——**类型靠目录前缀来识别**。我们按第 2 章那五种形态，建一个清晰的目录结构：

In [31]:
# 准备多源语料目录：按知识类型分子目录（目录名即类型线索）
!rm -rf ./output/l3-corpus
!mkdir -p ./output/l3-corpus/companies ./output/l3-corpus/people ./output/l3-corpus/meetings ./output/l3-corpus/decisions ./output/l3-corpus/notes
!ls ./output/l3-corpus

companies decisions meetings  notes     people


&emsp;&emsp;每个 Markdown 文件用 frontmatter 声明自己的 `type`，正文用 wikilink 连到相关的页。下面我们用 Jupyter 的 `%%writefile` 魔法命令，**每页写成一个独立 cell**（共 7 页：1 家公司、2 个人物、1 次会议、2 个决策、1 篇散记），它们之间用 wikilink 互相连接。（不用 `!cat <<EOF` 这种 heredoc——Jupyter 的 `!` 只把第一行当 shell，heredoc 正文会被当 Python 解析而报红；`%%writefile` 才是 Notebook 里写文件的正道。）注意 wikilink 全部**带目录前缀**（比如 `[[people/alex-chen]]` 而不是 `[[alex-chen]]`）——这一点至关重要，后面抽图谱边能不能成功，全看 wikilink 的目标 slug 和实际页 slug 是否精确匹配：

In [32]:
%%writefile ./output/l3-corpus/companies/harmony-ai.md
---
type: company
title: Harmony AI
---
# Harmony AI
Harmony AI 专注 AI 安全，CTO 是 [[people/alex-chen]]，增长由 [[people/jessica-wu]] 负责。
关键决策见 [[decisions/build-harmony-bridge]]。
CAC/NDR 毛利率指标尚未记录（知识盲区）。

Writing ./output/l3-corpus/companies/harmony-ai.md


In [33]:
%%writefile ./output/l3-corpus/people/alex-chen.md
---
type: person
title: Alex Chen
---
# Alex Chen
[[companies/harmony-ai]] 的 CTO，参与 [[meetings/q2-roadmap-review]]，主张优先 AI 安全。

Writing ./output/l3-corpus/people/alex-chen.md


In [34]:
%%writefile ./output/l3-corpus/people/jessica-wu.md
---
type: person
title: Jessica Wu
---
# Jessica Wu
负责 [[companies/harmony-ai]] 增长，牵头 [[decisions/build-harmony-bridge]]。

Writing ./output/l3-corpus/people/jessica-wu.md


In [35]:
%%writefile ./output/l3-corpus/meetings/q2-roadmap-review.md
---
type: meeting
title: Q2 路线图评审
---
# Q2 路线图评审
与 [[people/alex-chen]] 的 Q2 评审，讨论 [[decisions/build-harmony-bridge]] 与 [[decisions/q2-ai-safety]]。

Writing ./output/l3-corpus/meetings/q2-roadmap-review.md


In [36]:
%%writefile ./output/l3-corpus/decisions/build-harmony-bridge.md
---
type: decision
title: 推进 Harmony AI 增长备忘录
---
# 推进 Harmony AI 增长备忘录
为 [[companies/harmony-ai]] 准备增长备忘录，由 [[people/jessica-wu]] 牵头，关联 [[meetings/q2-roadmap-review]]，状态进行中。

Writing ./output/l3-corpus/decisions/build-harmony-bridge.md


In [37]:
%%writefile ./output/l3-corpus/decisions/q2-ai-safety.md
---
type: decision
title: Q2 优先 AI 安全
---
# Q2 优先 AI 安全
本季度优先 AI 安全，由 [[people/alex-chen]] 主张，来自 [[meetings/q2-roadmap-review]]，状态已确认。

Writing ./output/l3-corpus/decisions/q2-ai-safety.md


In [38]:
%%writefile ./output/l3-corpus/notes/growth-strategy.md
---
type: note
title: 增长打法散记
---
# 增长打法散记
关于 [[companies/harmony-ai]] 增长打法的初步散记，尚未成型。

Writing ./output/l3-corpus/notes/growth-strategy.md


In [39]:
# 确认 7 页都写好了
!echo "已写入语料文件："
!find ./output/l3-corpus -name '*.md' | sort

已写入语料文件：
./output/l3-corpus/companies/harmony-ai.md
./output/l3-corpus/decisions/build-harmony-bridge.md
./output/l3-corpus/decisions/q2-ai-safety.md
./output/l3-corpus/meetings/q2-roadmap-review.md
./output/l3-corpus/notes/growth-strategy.md
./output/l3-corpus/people/alex-chen.md
./output/l3-corpus/people/jessica-wu.md


&emsp;&emsp;真实场景里你会有更多文件——会议纪要、决策记录、人物档案，可能上百页。这里我们用一组精简的 7 页样例，但结构是一样的：每个文件声明 type、用带前缀的 wikilink 互连。特别注意 `companies/harmony-ai.md` 里我们故意写了一句"CAC/NDR 毛利率指标尚未记录（知识盲区）"——它的作用是：当你对这个库跑 `think` 时，大脑会在末尾的空白分析（Gaps）里如实把这个未记录的盲区标出来，而不是编一个数字。建好 brain-l3 后你可以自己跑一次 `think` 验证这一点。

### 3.3 import：写入时编译

&emsp;&emsp;语料准备好了，现在 import。这一步是 GBrain "编译式" 哲学的核心动作——**import 的瞬间，GBrain 就把 Markdown 编译成了 wiki 页、切块、向量化，并记录下每页正文里的 wikilink**。这和传统 RAG "查询时才算相似度" 完全不同：GBrain 把重活提前到写入时做完，查询时直接用现成的索引。要注意的是，import 这一步只是把页和它们的 wikilink 字段存好——真正把这些 wikilink 解析成 typed-edge 图谱（self-wiring 建边），还要单独跑一步 `gbrain extract links`，这就是下面 3.5 节的内容。

In [75]:
# import 多源语料：默认带 embedding（因为我们 init 时配了 embedding_model）
# 这一步会切块 + 向量化，把每页编译进大脑，是"写入时编译"
!gbrain import ./output/l3-corpus

#!gbrain import "/Users/mac/大模型资料/编译式RAG/GBrain/lesson-3/output/l3-corpus" --path "/Users/mac/大模型资料/编译式RAG/GBrain/lesson-3/output/brain-l3"

[gbrain phase] import.collect_files start dir=./output/l3-corpus strategy=markdown
[gbrain phase] import.collect_files done 15ms files=7
Found 7 markdown files
[import.files] 7/7 (100%) donerted=7 skipped=0 errors=0

Import complete (6.8s):
  7 pages imported
  0 pages skipped (0 unchanged, 0 errors)
  7 chunks created


&emsp;&emsp;import 跑完会给你一份小结：导入了几页（imported）、跳过了几页（skipped）、出错几页（errors）、创建了几个 chunk。我们这 7 页语料，正常会看到 `7 pages imported, 0 skipped, 7 chunks created`。这里有两个真实现象值得解释。

&emsp;&emsp;第一个，**进度里出现 "Skipped" 不一定是错误**。如果你重复 import 同一批文件，没有变化的页会被 skip（unchanged）——这是幂等设计，不是 bug。只有当 skip 后面跟着 errors 才需要警惕。第二个真实的坑，关于 embedding 阶段的网络抖动：

> **【踩坑预警 · embedding 阶段的偶发网络错误】**　import 时会逐页调用 embedding provider 做向量化。如果遇到网络抖动（比如 DashScope 偶尔会报 `unknown certificate verification error` 或 `Incorrect API key`），GBrain 会跳过那一页，结果是 imported 比总页数少 1-2 页。**正确做法**：① 如果只是偶发抖动，重新跑一遍 import（已成功的页会 skip-unchanged，失败的页会补上）；② 如果整批都失败，多半是 key 真的不可用，换一个可用的 provider——比如 DashScope 不稳时换 `openai:text-embedding-3-small`（走 OpenAI 兼容代理），重新 `init --embedding-model openai:text-embedding-3-small` + import。**这也是为什么我们强调凭证分离**——LLM 用 DeepSeek、embedding 用 DashScope 或 OpenAI，它们是不同的 key，任一不稳都可以独立切换。

### 3.4 embed：确认向量化完整

&emsp;&emsp;正常情况下 import 已经把 embedding 做完了。但有时候——比如你先用 `--no-embed` 导入、或者 import 中途 embedding 失败补跑——你需要手动触发一次向量化补全。这就是 `embed` 命令：

In [76]:
# embed --all：对所有还没向量化的页补做 embedding（import 已嵌入时这步会很快跳过）
!gbrain embed --all

[embed.pages] 7/7 (100%)Embedded 7 chunks across 7 pages
[embed.pages] 7/7 (100%) done


&emsp;&emsp;`embed` 有几个模式：`--all` 对所有页、`--stale` 只对内容变了的页、也可以传单个 slug。在我们这个场景里，因为 import 已经嵌入过了，`embed --all` 跑起来会发现没什么要做的、很快返回。但记住这个命令——当你后面用 `--no-embed` 快速导入大批文件、再统一补 embedding 时，它就是你的工具。

In [77]:
# Tier 1 补充验证：stats 看 Embedded 数，确认向量化覆盖
!gbrain stats

Pages:     7
Chunks:    7
Embedded:  7
Links:     0
Tags:      0
Timeline:  0

By type:
  decision: 2
  person: 2
  note: 1
  meeting: 1
  company: 1


&emsp;&emsp;这次看 `stats` 的重点是 **Embedded 这一行**——我们这 7 页应该看到 `Embedded: 7`，等于 Chunks 数，说明每页都被向量化了。如果 Embedded 远小于 Chunks，说明有页没嵌上，回去查 embedding 那一步（参考上面的网络抖动踩坑框）。Embedded 数等于 Chunks，就是"向量检索能力已就绪"的硬凭证。

### 3.5 extract links：抽取图谱边

&emsp;&emsp;这一步很容易被忽略，但它是图谱能不能用起来的关键。前面 import 把页都导进来了，但你现在去看 `stats` 会发现 **Links 是 0**——图谱一条边都没有。很多人到这里会以为是 wikilink 写错了，其实不是：**import 只负责把页导进来，关系边要再单独跑一步 `extract links` 才会建出来**。

In [78]:
# extract links：从已导入的页里抽取 typed edges（关系边）
# --source db 表示从数据库里已编译的页抽，纯解析 wikilink，零 LLM
!gbrain extract links --source db

[extract.links_db] 7/7 (100%) done
Links: created 12 from 7 pages (db source)

Done: 12 links, 0 timeline entries from 7 pages


&emsp;&emsp;这一步就是 **self-wiring 真正发生的时刻**——它扫描每页正文里的 wikilink，根据目标 slug 把页和页之间连起来，建出 typed edges（带类型的边，比如"谁创立了哪家公司""谁参加了哪个会议"）。整个过程零 LLM，纯靠解析 wikilink。我们这 7 页跑完，正常会看到 `Links: created 12 from 7 pages`——12 条边，正是我们在语料里用带前缀 wikilink 连出来的那些关系。

### 3.6 验收三件套：list / stats / graph

&emsp;&emsp;到这一步，我们要回收第 1 章钉下的痛点一了——"知识散落各处"。现在它们被编译进了一个大脑、连成了一张图谱。我们用三个命令验收：`list` 看页、`stats` 看规模、`graph` 看图谱。这是本章的 Tier 2 综合验收。

&emsp;&emsp;先说一件诚实的事：**你跑出来的数字，是你自己语料的数字，不是某个标准答案**。我们这个样例语料是 7 页 / 12 条边，你导入自己的上百页知识会得到完全不同的数字。所以下面命令的输出，重点是看"结构对不对"，而不是"数字等不等于某个值"。

In [79]:
# list：列出大脑里所有页（slug / type / 日期 / 标题）
!gbrain list

companies/harmony-ai	company	2026-06-26	Harmony AI
decisions/build-harmony-bridge	decision	2026-06-26	推进 Harmony AI 增长备忘录
decisions/q2-ai-safety	decision	2026-06-26	Q2 优先 AI 安全
meetings/q2-roadmap-review	meeting	2026-06-26	Q2 路线图评审
notes/growth-strategy	note	2026-06-26	增长打法散记
people/alex-chen	person	2026-06-26	Alex Chen
people/jessica-wu	person	2026-06-26	Jessica Wu


&emsp;&emsp;`list` 会把每一页列出来：slug、type、创建日期、标题。你应该看到你 import 的页都在，type 也对（decision / person / note / meeting 各归各类）。这验证了一件事：你的 schema 映射生效了——目录前缀和 frontmatter 的 type 字段正确地把每页归了类。

In [80]:
# stats：大脑规模总览（页数 / chunk / 向量 / 链接 / 按类型分布）
!gbrain stats

Pages:     7
Chunks:    7
Embedded:  7
Links:     12
Tags:      0
Timeline:  0

By type:
  decision: 2
  person: 2
  note: 1
  meeting: 1
  company: 1


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171258524.png" width=72%></div>

&emsp;&emsp;`stats` 给你大脑的体检报告：Pages（页数）、Chunks（切块数）、Embedded（向量数）、Links（图谱边数）、以及 By type 的类型分布。我们这 7 页语料，跑完应该看到 `Pages: 7 / Chunks: 7 / Embedded: 7 / Links: 12`——注意 Links 现在是 12 而不是 0，因为我们上一步跑了 `extract links`。这就是你的大脑"长出来"的证据。你自己的语料会是你自己的数字——可能是几十页、几百页。**7 页只是为了让你看清每一步的机制**；等下到 3.7 节，我们会真的导入一份上百页的真实多源语料，亲眼看 self-wiring 在真实规模上抽出上千条边。

In [81]:
# graph：看某一页的图谱连接（extract links 建出的 typed edges）
!gbrain graph companies/harmony-ai --depth 1

[
  {
    "slug": "companies/harmony-ai",
    "title": "Harmony AI",
    "type": "company",
    "depth": 0,
    "links": [
      {
        "to_slug": "people/alex-chen",
        "link_type": "mentions"
      },
      {
        "to_slug": "people/jessica-wu",
        "link_type": "mentions"
      }
    ]
  },
  {
    "slug": "people/alex-chen",
    "title": "Alex Chen",
    "type": "person",
    "depth": 1,
    "links": [
      {
        "to_slug": "companies/harmony-ai",
        "link_type": "mentions"
      },
      {
        "to_slug": "meetings/q2-roadmap-review",
        "link_type": "mentions"
      }
    ]
  },
  {
    "slug": "people/jessica-wu",
    "title": "Jessica Wu",
    "type": "person",
    "depth": 1,
    "links": [
      {
        "to_slug": "companies/harmony-ai",
        "link_type": "mentions"
      }
    ]
  }
]


&emsp;&emsp;`graph <slug>` 让你从某一页出发遍历它的图谱连接。注意这个命令**需要一个 slug 参数**（不能裸跑 `gbrain graph`）。跑 `companies/harmony-ai` 你会看到它向外连了几个节点——CTO、增长负责人、关键决策，这些就是 `extract links` 根据 wikilink 抽出来的 typed edges，整个过程没有调用任何 LLM。

> **【常见误区 · Links=0 先检查是不是漏跑了 extract links】**　如果你跑 `stats` 发现 Links 是 0、`graph` 返回的 links 是空数组，**第一件事是检查你是不是漏跑了 `extract links`**——这是最常见的原因。import 只导页不抽边，边必须单独跑 `extract links --source db` 才会建出来（就是 3.5 那一步）。**第二个可能原因**：wikilink 的目标 slug 没和实际页 slug 精确匹配。比如你写 `[[alex-chen]]`，但实际页 slug 是 `people/alex-chen`，前缀对不上，这条边就抽不出来。**正确做法**：① 先确认跑过 `extract links`；② 再确认 wikilink 都带了目录前缀（写 `[[people/alex-chen]]` 而不是 `[[alex-chen]]`）。这正是第 2 章那条"目录前缀不能漏"铁律的真实价值——wikilink 只有带对目录前缀，抽边时才连得上目标页。

### 3.7 真实规模演示：154 页语料

&emsp;&emsp;前面的 7 页语料是为了让你看清每一步机制——但一个真正用起来的大脑长什么样？这一节我们就导入一份**真实规模** 的多源语料，亲眼看 self-wiring 在真实数据上的威力。

&emsp;&emsp;课程附带了一份真实的多源语料 `datasets/vc-brain-corpus/`，它是一个风投合伙人（YC 风格）的知识世界，共 **154 个文件**，跨五个源：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>vc-brain-corpus：一份真实多源语料</font></p>
<div class="center">

| 源目录 | 文件数 | 内容 |
|--------|--------|------|
| `email/` | 46 | 邮件往来（引荐、董事会、融资讨论） |
| `slack/` | 55 | Slack 频道讨论（周会、投资组合更新） |
| `calendar/` | 36 | 日历事件（GP 会议、面试、check-in） |
| `people/` | 9 | 人物档案（Garry Tan、Sam Altman 等） |
| `companies/` | 8 | 公司档案（Y Combinator、OpenAI、Stripe 等） |

</div>

&emsp;&emsp;每个文件都带 frontmatter + 跨源 `[[wikilink]]`——一封 email 会链到 `[[people/garry-tan]]` 和 `[[companies/openai]]`，一条 slack 讨论会链到参与的人和公司。这正是 self-wiring 的燃料。我们把它导入一个**新的** 大脑（和你前面的 7 页玩具库分开，用 `./output/brain-vc`）：

In [82]:
# 新建一个带 embedding 的大脑，专门放这份真实语料
# 不带 --force：§3.1 已清掉全局 embedding_disabled，且这是全新库，init 会直接按 --embedding-model 配好
!gbrain init --path ./output/brain-vc --embedding-model dashscope:text-embedding-v3

Setting up local brain with PGLite (no server needed)...
  Embedding: dashscope:text-embedding-v3 (1024d)
  Expansion: openai:gpt-5.2
[init] Using schema pack: gbrain-base-v2 (override with --schema-pack <name>)

Brain ready at ./output/brain-vc
154 pages. Engine: PGLite (local Postgres).

Existing brain detected. To wire up the v0.10.3 knowledge graph:
  gbrain extract links --source db        (typed link backfill)
  gbrain extract timeline --source db     (structured timeline backfill)
  gbrain stats                            (verify links > 0)

When you outgrow local: gbrain migrate --to supabase

--- GBrain Mod Status ---
Skills: 51 loaded
GStack: not found
  Install GStack for coding skills:
  git clone https://github.com/garrytan/gstack.git ~/.claude/skills/gstack
  cd ~/.claude/skills/gstack && ./setup
Resolver: skills/RESOLVER.md
Soul audit: run `gbrain soul-audit` to customize agent identity
Retrieval reflex: on by default (entity pointers injected per turn)
  Install the pol

In [51]:
# 导入 154 个文件——这一步会慢一些（约 2 分钟），因为它在给 154 页逐一算向量
!gbrain import datasets/vc-brain-corpus

[gbrain phase] import.collect_files start dir=datasets/vc-brain-corpus strategy=markdown
[gbrain phase] import.collect_files done 17ms files=154
Found 154 markdown files
[import.files] 154/154 (100%) donerted=154 skipped=0 errors=0

Import complete (102.7s):
  154 pages imported
  0 pages skipped (0 unchanged, 0 errors)
  154 chunks created


&emsp;&emsp;真实输出（单次实测，导入耗时随网络 / 机器波动，不是固定值）：

```
Import complete (122.4s):
  154 pages imported
  154 chunks created
```

&emsp;&emsp;154 页、约两分钟——慢是因为每页都要调一次 embedding API 算向量，这是真实规模导入该有的体感。导入完，照例单独跑 `extract links` 抽边：

In [52]:
# self-wiring：在真实语料上抽 typed edges
!gbrain extract links --source db

[extract.links_db] 154/154 (100%) done
Links: created 1172 from 154 pages (db source)

Done: 1172 links, 0 timeline entries from 154 pages


```
Links: created 1172 from 154 pages (db source)
```

&emsp;&emsp;**1172 条边**。回头看你那个 7 页玩具库才 12 条——同一个零 LLM 的 self-wiring 机制，喂给它真实规模的互链语料，就自动织出了上千条 typed edges。跑 `stats` 看全貌：

In [53]:
!gbrain stats

Pages:     154
Chunks:    154
Embedded:  154
Links:     1172
Tags:      12
Timeline:  0

By type:
  slack-thread: 55
  email: 46
  calendar-event: 36
  person: 9
  company: 8


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171259828.png" width=50%></div>

&emsp;&emsp;154 页全部向量化、**1172 条边**、五种类型各归各类——一个真实多源大脑的体检报告。

&emsp;&emsp;停一下，把这份体检报告和第 2 章的 schema 扣起来看，它正是「现成包够用」的实锤。回头看这五种类型：`person` / `company` / `email` 是 `gbrain-base-v2` 那 15 个规范类型里**本来就有的**，直接归类；`slack-thread` 不在规范名里，但它是 `slack` 的**别名**（就是第 2 章讲的别名归一机制），包自动把它当 `slack` 认；`calendar-event` 则是 15 类和别名表里都**没有的越界类型**——正是第 2 章结尾说的那种「标准包覆盖不到的自定义类型」（当时我们拿 `meeting` 举的例），GBrain 不把它当注册类型，但**照你写的原样存成一页**，检索、连边一点不耽误。也就是说，这 154 页里规范类型、别名、越界自定义类型三种全齐了，可你**一行自定义 schema 都没写**，`gbrain-base-v2` 就把它们全吃下、各归各类。连那 1172 条边的类型（`works_at` / `mentions` 这些）也是默认包靠它内置的 14 个关系动词在导入后自动抽出来的。这就把第 2 章那句结论坐实了：**绝大多数场景开箱即用，自定义 schema 只在你想让越界类型也受类型校验、或想要更精确的关系动词时才需要**。

&emsp;&emsp;我们从某个人物节点出发看图谱：

In [54]:
# 从 Garry Tan 这一页出发，看它连到哪
!gbrain graph people/garry-tan --depth 1

[
  {
    "slug": "people/garry-tan",
    "title": "Garry Tan",
    "type": "person",
    "depth": 0,
    "links": [
      {
        "to_slug": "companies/y-combinator",
        "link_type": "works_at"
      }
    ]
  },
  {
    "slug": "companies/y-combinator",
    "title": "Y Combinator",
    "type": "company",
    "depth": 1,
    "links": [
      {
        "to_slug": "people/dalton-caldwell",
        "link_type": "mentions"
      },
      {
        "to_slug": "people/garry-tan",
        "link_type": "mentions"
      },
      {
        "to_slug": "people/michael-seibel",
        "link_type": "mentions"
      }
    ]
  }
]


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171259753.png" width=50%></div>

&emsp;&emsp;输出是真实的 typed edges：`people/garry-tan` 通过 `works_at` 连到 `companies/y-combinator`，再从 y-combinator 往外 `mentions` 到 `people/dalton-caldwell`、`people/michael-seibel`……一张真实的人—公司—事件关系网，全是零 LLM 抽出来的。最关键的一步——在这份真实语料上跑 `think`，看跨源综合：

In [56]:
# 中文提问，跨 154 页英文多源综合（注意：语料是英文，但你照样可以用中文问）
!gbrain think "Garry Tan 最近参与了哪些公司和会议？有什么待办？" --model deepseek:deepseek-v4-flash

# Garry Tan 最近参与了哪些公司和会议？有什么待办？

{
  "answer": "Garry Tan 最近参与的公司和会议如下：\n\n## 公司\n- **Y Combinator**：作为 Managing Director，参与了多项会议和邮件沟通，包括 YC 董事会会议准备、季度运营计划展示、投资人更新草稿讨论等。[email/email-2026-03-29-0014-y-combinator][email/email-2026-04-09-0025-y-combinator][slack/slack-deal-flow-2026-04-24-0039][slack/slack-portfolio-updates-2026-04-16-0029]\n- **OpenAI**：参与 OpenAI 董事会会议，展示 Q1 运营计划，并参与 Series B 条款书修改。[email/email-2026-03-16-0044-openai][email/email-2026-03-20-0042-openai][slack/slack-weekly-sync-2026-04-13-0026]\n- **Scale AI**：参加 GP 会议讨论 Scale AI 的 Series B 估值基准；领导 YC x Scale AI 技术审查会议，等待其资源分配审批。[calendar/calendar-2026-04-10-0024-gp-meeting][calendar/calendar-2026-04-05-0021-technical-review]\n- **Anthropic**：接收 Anthropic 董事会���议行动项邮件，参加 YC x Anthropic 创始人检查会议（担任记录人）。[email/email-2026-04-08-0022-anthropic][calendar/calendar-2026-03-20-0029-founder-check-in]\n- **Stripe**：在 Slack 中发布 Stripe 投资组合更新（ARR 里程碑、OKR 对齐），与 Julia Hartz 讨论。[slack/slack-portfolio-updates-2026-04-16-0029][slack/slack-

&emsp;&emsp;真实输出——一段结构化的中文答案，"参与的会议""待办事项""Gaps"三段分明，每条都带 `[源/页]` 引用（注意 Citations 数每次略有浮动，因为 think 要调一次 LLM 生成）：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171935966.png" width=50%></div>

&emsp;&emsp;一个问题，think 跨 **40 页** email / slack / calendar 综合出一段带引用的中文答案，还附上"还缺什么"的空白分析（这里的 `40` 是 `think` 默认的综合页预算——库够大、相关页够多时基本都会用满，所以你在不同查询里常看到底部 `Pages: 40`；库小于这个数时就显示实际页数，比如前面 7 页玩具库是 `Pages: 7`）。这就是 self-wiring 图谱 + 混合检索 + LLM 综合三者在真实规模上叠加的效果。

> **【真实坑 · 语料是英文，但你能用中文问】**　这份语料的内容是英文（邮件、Slack 都是英文），但你刚才看到——**用中文提问照样能召回、还用中文作答**。这是因为本节配置的 embedding 模型 `dashscope:text-embedding-v3` 是多语言的，向量臂能跨语言桥接：中文问题被编码成向量后，照样匹配到语义相近的英文页。两点边界要知道：① 跨语言时相关性分数通常会低一些（中文 query 命中同一页的分数，往往低于直接用英文 query），所以语言对齐时召回质量更好；② 跨语言能力来自**向量臂**——`search` / `query` / `think` 默认都带向量，所以都能跨语言；其中 `query` / `think` 还多一层 LLM 查询扩展（能把中文问题扩写成多个变体），跨语言召回通常更稳一些。当环境里没配可用 embedding provider、或 embedding 调用失败、检索退成纯 BM25 词面匹配时，才会跨不了语言（中文词匹配不上英文正文）。

&emsp;&emsp;对比一下就看出真实规模的意义：7 页玩具库教你看清每一步机制，154 页真实语料让你看到这套机制的真正价值——**12 条边 vs 1172 条边，单页零碎事实 vs 跨 40 页的综合答案**。self-wiring 和 think 的威力，要在真实规模上才看得真切。

&emsp;&emsp;从这里起，**这个 154 页的真实大脑就是我们后续的工作主库**——第 4 章的 `think` 会前综合、第 5 章接进 Claude Code 当长记忆，都在它上面做。前面那个 7 页小库已经完成了它的使命：在它身上你看清了建库、编译、self-wiring 的每一步。接下来，我们换上这个真实规模的大脑，去体验它在日常工作里到底怎么用——毕竟你将来给自己领域建的大脑，是几百上千页的真实知识，而不是 7 页玩具。

&emsp;&emsp;本章我们走完了第一个核心里程碑：从 `init --embedding-model` 建带向量的库，到 `import` 写入时编译，到 `embed` 补全向量化，到 `list / stats / graph` 三件套验收，最后用一份 154 页真实语料看到了 self-wiring 在规模上的威力。**痛点一回收完成**——你的知识不再散落，它们被编译进了一个可检索、有图谱的大脑。但现在你只能"搜"它（返回页面）。下一章，我们让它从"搜页面"升级到"替你读完多页、综合成答案"——这就是 `think` 的舞台。

---

## <center>第 4 章：查询大脑（综合层）</center>

&emsp;&emsp;这是本节第二个核心落地章。第二节我们已经把 `search` 到 `think` 的范式跨越讲透了——`search` 给你页面，`think` 替你读完页面再写答案。这一章我们把那套认知真正落到自己的大脑上：先用 `search` 感受"找到了页但还得自己读"的痛点，再用 `think` 看它怎么直接给你成文答案 + 引用 + 空白分析。这一章会回收第 1 章的痛点二。

&emsp;&emsp;这一章层层递进：先用 `search` vs `think` 的对照让你撞见"找到页还得自己读"的痛点，再把会前综合的完整效果摆出来，最后拆解末尾的 Gaps 空白分析、以及单源 vs 跨源的价值差异。

### 4.1 search 取页 vs think 综合

&emsp;&emsp;我们继续用上一章建好的 154 页 `brain-vc`——3.7 节「真实规模演示」建它时活动库就已经切到了 `brain-vc`，正好接着用，本章不用再切库。这里要点破一个**最容易踩的坑**：`think` / `search` / `stats` / `list` / `graph` 这些读命令操作的永远是"**活动库**"——也就是你最近一次 `gbrain init` 选定的那一个，它们**根本不接受 `--path` 参数**（只有 `init` 才认 `--path`）。你在 `think` 后面写 `--path ./output/xxx` 是**无效的**，它会被当成问题文本忽略掉，命令照样打在当前活动库上。所以本章所有读命令都直接打在当前活动库 `brain-vc` 上，不写 `--path`。

&emsp;&emsp;先用 `search` 查我们上一章建好的 154 页大脑。这里先把第二节讲过的一对命令辨析清楚：`search` 和 `query` **默认都是混合检索**（HNSW 向量 + BM25 + RRF 融合，都带语义向量）——它们的区别不是"带不带向量"，而是控制粒度：`query` 是完整可控版，能开关 LLM 查询扩展；`search` 是轻量版，默认不做查询扩展、成本更低（在没有可用 embedding provider、或 embedding 调用失败等情况下，两者才会退成纯关键词）。本节这一步我们用轻量的 `search`，就是为了让你感受"检索把相关页都召回了、但你还得自己打开读、自己综合"的痛点——这个痛点跟检索走不走向量无关，是所有"检索给页面"范式的共同天花板。`search` 返回匹配的页面列表和相似度评分：

In [57]:
# search：轻量混合检索（向量 + BM25 + RRF，默认不做查询扩展），返回页面列表 + 相似度评分
!gbrain search "OpenAI Series B 估值"

[1.1264] companies/openai -- # OpenAI

**Sector:** AI Research / Products
**Founded by:** [[people/sam-altman]]

OpenAI is an AI 
[1.0395] companies/scale-ai -- # Scale AI

**Sector:** AI Data / Infrastructure
**Founded by:** [[people/alexandr-wang]]

Scale AI 
[1.0306] people/sam-altman -- # Sam Altman

**Role:** CEO, OpenAI
**Organizations:** [[companies/openai]] [[companies/y-combinator
[0.9199] people/alexandr-wang -- # Alexandr Wang

**Role:** CEO & Founder, Scale AI
**Organizations:** [[companies/scale-ai]]

Alexan
[0.8896] email/email-2026-03-20-0042-openai -- # Follow-up: OpenAI Series B

**Email Chain · 4 messages · 2026-03-20**

---

**Most Recent — 2026-0
[0.8259] email/email-2026-04-18-0033-openai -- # Investor Update QQ1 2026

**Email Chain · 7 messages · 2026-04-18**

---

**Most Recent — 2026-04-
[0.8122] email/email-2026-03-17-0040-openai -- # Action Items from OpenAI Board Meeting

**Date:** 2026-03-17
**From:** [[people/alexandr-wang]]
**
[0.8017] email/email-2026-03-

&emsp;&emsp;真跑出来你会看到几行：每行是一个 `[相似度] slug -- 摘要`。比如 `[1.1264] companies/openai`、`[1.0395] companies/scale-ai`。这很有用——它确实把相关的页都找出来了，按相似度排好序。但请你停下来感受一下这里的痛点：

> **【痛点激活 · 找到页 ≠ 准备好了】**　`search` 告诉你"这几页和你的问题相关"，但要真正回答"见 OpenAI 团队前我该知道什么"，你还得**自己打开这几页、读完、在脑子里综合**。找到对的页，只是工作的开始，不是结束。这就是传统检索的天花板——它帮你定位，但不帮你思考。

&emsp;&emsp;现在我们换 `think`。同一个问题域，但这次让大脑替我们读完再写答案：

In [59]:
# think：多页综合问答，输出成文答案 + 逐条引用 + 末尾空白分析
# 用 --model deepseek:deepseek-v4-flash 指定合成模型（DeepSeek 做综合的 LLM）
# 关键：model id 必须带 deepseek: 前缀，否则会被默认补成 anthropic: 前缀导致 unknown_model
!gbrain think "OpenAI 的 Series B 进展如何？估值多少？Garry Tan 在跟进什么？请使用中文回复" --model deepseek:deepseek-v4-flash

# OpenAI 的 Series B 进展如何？估值多少？Garry Tan 在跟进什么？请使用中文回复

## OpenAI Series B 进展与估值

OpenAI 的 Series B 融资正在积极推进中。根据 Garry Tan 的邮件记录，2026年3月20日的投资意向书（term sheet）提出估值上限为 **4.88 亿美元（pre-money）**，附带 18% 的跟投权 [email/email-2026-03-20-0042-openai]。到 2026年4月18日，估值上限上调至 **5.46 亿美元（pre-money）**，跟投权也升至 20% [email/email-2026-04-18-0033-openai]。这表明条款在谈判中有所优化。

其他内部信息显示，OpenAI 的融资进度良好：2026年3月17日的周报提到“融资时间线进展顺利，比计划提前 39%” [slack/slack-weekly-sync-2026-03-17-0041]。2026年4月20日，Brian Chesky 在 #market-intel 中分享了 OpenAI 的 Q2 数据强劲，ARR 同比增长 47% [slack/slack-market-intel-2026-04-20-0036]。

## Garry Tan 的跟进动态

Garry Tan 深度参与了 OpenAI Series B 的跟进工作：
- **审查并修改 term sheet**：他在 3月20日和4月18日分别提出了 valuation cap、pro-rata 权利和董事会组成等修改建议 [email/email-2026-03-20-0042-openai] [email/email-2026-04-18-0033-openai]。
- **参与董事会会议**：2026年3月16日的一封邮件显示，Garry Tan 将在即将到来的 OpenAI 董事会会议上展示 2026 年第一季度运营计划 [email/email-2026-03-16-0044-openai]。
- **监控运营指标**：2026年4月13日，他在 #weekly-sync 频道发布了 OpenAI 的每周信号，包括 ARR、流失率、NPS 和现金流期限等关键指标 [slack/slack

&emsp;&emsp;这一条命令真跑下来，输出和 `search` 完全是两个量级的东西。它不再是页面列表，而是一篇结构化的成文答案——分了"进展""估值""Garry Tan 在跟进什么""冲突""Gaps"几段，每个论断后面都跟着方括号里的来源页 slug（如 `[slack/slack-weekly-sync-2026-03-17-0041]`、`[email/email-2026-03-20-0042-openai]`），底部还有一行 `Model / Pages / Takes / Graph / Citations` 的元信息（这个 154 页的库会看到 `Pages: 40`——这是 `think` 默认的综合页预算，库够大、相关页够多时基本拉满；`Citations: 9` 是这次综合实际引用的来源处数，综合答案是 LLM 生成的，引用数每次略有浮动是正常的）。**痛点二在这里就回收了**：你不用再自己读页，`think` 替你读完并综合好了——它甚至主动标出了一段"冲突"：同一个 OpenAI 的 ARR 指标，在不同周报里从 5400 万、3600 万一路跌到 1800 万美元，口径自相打架。这种跨页才看得出的矛盾，正是下一节要讲的重点。

> **【踩坑预警 · think 指定 DeepSeek 模型必须带 provider 前缀】**　模型 id 一定要写全 `deepseek:deepseek-v4-flash`，**前面的 `deepseek:` 前缀不能省**。如果你只写 `--model deepseek-v4-flash`（裸名），gbrain 会默认替你补上 `anthropic:` 前缀、当成 anthropic 系模型去查注册表，结果报 `unknown_model`。带上 `deepseek:` 前缀后 `--model deepseek:deepseek-v4-flash` 就完全有效——这一点我们已经实测过。你也可以用环境变量前缀的等价写法 `GBRAIN_MODEL=deepseek:deepseek-v4-flash gbrain think "..."`，两种写法都对，关键都在于**那个 `deepseek:` 前缀**。另外，`think` 的元信息行偶尔会出现 `Warnings: LLM_OUTPUT_NOT_JSON`——这是合成模型 `deepseek-v4-flash` 偶发把答案包进一层 JSON 没闭合所致，gbrain 会自动用正则兜底把引用抓出来、`Citations` 数照样准；介意输出不够干净的话，重跑一遍通常就恢复成规整的 Markdown。

### 4.2 会前综合演示

&emsp;&emsp;刚才问的是 OpenAI 一家公司。现在我们把问题放大到**整个投资组合**——这正是 `think` 最该上场的"会前综合"场景：开一次 GP 会议或董事会前，你想一次性搞清楚"手上所有公司、所有会议里，最重要的决策和待办是什么，有没有冲突和盲区"。一条命令：

In [62]:
# 会前综合：一个横跨整个投资组合的全局问题，看 think 的综合上限
!gbrain think "Garry Tan 最近在投资组合上有哪些关键决策和待办行动项？涉及哪些公司和人？有没有盲区？" --model deepseek:deepseek-v4-flash

[chat(deepseek:deepseek-v4-flash)] unknown certificate verification error


```
# Garry Tan 最近在投资组合上有哪些关键决策和待办行动项？涉及哪些公司和人？有没有盲区？

## 关键决策
1. OpenAI 条款修改：pre-money 估值上限 $4.88 亿、18% pro-rata、董事会构成
   [email-2026-03-20-0042-openai]
2. Scale AI 条款修改：估值上限 $7.34 亿、26% pro-rata [email-2026-03-15-0037-scale-ai]
3. 资源配置：批准 YC×Anthropic（Q3 上线）、YC×Scale AI（Q4 上线）的集成投入
   [calendar-2026-03-20-0029-founder-check-in]

## 待办行动项
- 跟进 OpenAI×Stripe 合作对 Stripe 持仓的影响 [slack-weekly-sync-2026-04-13-0026]
- YC 董事会准备：提交 Q4 运营计划（ARR 5400 万、招聘、战略定位）
  [email-2026-03-29-0014-y-combinator]

## 潜在盲区
1. 竞争风险被低估：YC×Airbnb、Stripe×YC、Scale AI×Instacart 多处企业级竞争，缺系统应对
2. 高流失率预警：OpenAI 流失率 3.4% 高于行业均值 2.1% [slack-weekly-sync-2026-04-13-0026]
3. 个人带宽风险：Garry Tan 同时领多项集成规划 + 多家条款谈判，精力分散

## Gaps
- 各合作项目（Rippling / Stripe 等）的时间表和里程碑
- 对高流失率（OpenAI 3.4%）的深入分析和应对
- Garry Tan 对资源分配和风险管理的优先级排序

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 29
```

&emsp;&emsp;请你逐段看这个输出的结构，它正是 `think` 综合能力的完整展现：**关键决策** 按公司列出（OpenAI 估值 4.88 亿美元、Scale AI 7.34 亿美元的条款修改，YC×Anthropic / YC×Scale AI 的资源配置审批），每条都带 `[email-...]`、`[calendar-...]` 来源；**待办行动项** 把散在各处的 follow-up 收拢成一张清单（跟进 OpenAI×Stripe 合作、YC 董事会 Q4 计划筹备）；**潜在盲区** 是 `search` 永远给不了你的——它跨页推理出了"OpenAI 流失率 3.4% 高于行业均值 2.1% 却没人深究""Garry Tan 同时领多个集成谈判、个人带宽吃紧"这类隐患；末尾 **Gaps** 段再诚实列出大脑里压根没有的信息。底部 `Citations: 29` 告诉你这一个答案综合了 29 处引用、跨 40 页——这就是会前综合的威力，也是 `search` 给页面列表永远做不到的。

### 4.3 空白分析：Gaps 段的价值

&emsp;&emsp;很多人看 `think` 的输出，眼睛只盯着正文答案。但老手最看重的是末尾那段 **Gaps（知识盲区）**。为什么？因为正文答案告诉你"大脑知道什么"，而 Gaps 告诉你"大脑诚实承认自己不知道什么"——这后者才是会前准备真正要补的功课。

&emsp;&emsp;我们专门问一个会暴露盲区的具体问题——Anthropic 的财务数字。这家公司大脑里有零散提及，但数据不全、还互相打架，正好看 `think` 怎么诚实处理：

In [63]:
# 问一个大脑里数据不全、还互相矛盾的具体问题
!gbrain think "Anthropic 的最新估值、ARR 和烧钱率的具体数字分别是多少？" --model deepseek:deepseek-v4-flash

# Anthropic 的最新估值、ARR 和烧钱率的具体数字分别是多少？

# Anthropic Latest Metrics

**Latest ARR:** $29M (as of 2026-04-18) [email/email-2026-04-18-0034-anthropic]

**Burn Multiple:** 1.9 (as of 2026-03-21) [email/email-2026-03-21-0004-anthropic]. Note: This is a multiple (burn ÷ net new ARR), not a dollar‑denominated monthly burn rate. A dollar burn rate is not available in the provided data.

**Valuation:** Not available. The only valuation data is a $239M pre‑money cap from a Series B term sheet dated 2026-03-17, which is a cap and not a current valuation [email/email-2026-03-17-0036-anthropic].

## Conflicts

The brain contains multiple different ARR figures across dates and sources: $6M (2026-04-11), $29M (2026-04-18), $33M (2026-04-01), $37M (2026-03-31), $40M (2026-03-30), $46M (2026-03-18), $55M (2026-04-08), $68M (2026-03-15), $80M (2026-03-21). These may represent different time periods or definitions. The latest by date is $29M, but no reconciliation is available.

## Gaps

- Current valuati

```
# Anthropic 的最新估值、ARR 和烧钱率的具体数字分别是多少？

## 估值
仅一条线索：3/17 Series B 条款清单里 pre-money 估值上限 $2.39 亿
[email/email-2026-03-17-0036-anthropic]。没有更近期数据。

## ARR
大脑里有一连串互相矛盾的数字（按时间倒序）：
  2026-04-18 $29M | 2026-04-11 $6M | 2026-04-08 $55M | 2026-04-01 $33M
  2026-03-31 $37M | 2026-03-21 $80M | 2026-03-15 $68M
最接近当前的是 4/18 的 $29M，但与一周前的 $55M 显著矛盾。

## 烧钱率
仅一个间接指标：3/21 邮件提到 Burn Multiple = 1.9
[email/email-2026-03-21-0004-anthropic]，无美元口径数字。

## Gaps
- Anthropic 的最新估值（2026 年 4 月之后）
- 以美元计价的烧钱率（仅有一个 Burn Multiple 间接指标）
- 为何 ARR 在几周内从 $80M 剧烈波动到 $29M（口径不同或存在错误）

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 9
```

&emsp;&emsp;看 `think` 怎么处理这堆残缺又矛盾的数据：估值它只找到一条 3/17 的 2.39 亿美元、并明说"没有更近期数据"；ARR 它没有硬编一个数字，而是把一连串互相矛盾的数字（从 8000 万一路缩到 2900 万美元）按时间列出来、点名"波动剧烈，可能口径不同或存在错误"；烧钱率它老实说"只有一个 Burn Multiple = 1.9 的间接指标，没有美元数字"。这就是大脑的诚实——它不会编一个数字糊弄你，而是明确告诉你"这块数据要么没有、要么对不上，你见面前得自己核实"。Gaps 通常有三种类型，下面这张表帮你判读：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>空白分析（Gaps）的三种类型</font></p>
<div class="center">

| Gap 类型 | 含义 | 你该做什么 |
|---------|------|-----------|
| 无数据 | 大脑里根本没有这块信息 | 见面/决策前自己去补这块知识 |
| 已过期 | 有数据但太旧（如 6 周没更新） | 确认信息是否还成立，更新大脑 |
| 自相矛盾 | 多页之间说法冲突 | 弄清楚哪个是对的，消解矛盾 |

</div>

&emsp;&emsp;Gaps 段的存在，是 `think` 区别于"会一本正经胡说"的普通 RAG 的根本。它宁可告诉你"我不知道"，也不编。这种诚实，在你拿它做真实决策准备时格外重要——它让你清楚知道哪些功课还得自己补。

### 4.4 单源 vs 跨源综合

&emsp;&emsp;`think` 的威力在哪个场景最明显？答案是**跨源问题**。如果你问的问题只涉及单一一页就能答的（比如"Sam Altman 是谁"），`think` 和直接读那一页差别不大。但当你问的问题需要把好几类知识（人物 + 会议 + 决策）拼起来才能答时，`think` 的跨页综合才真正发力。我们做个对照实验：

In [64]:
# 单源问题：只涉及一个 person 页，think 的综合空间有限
!gbrain think "Sam Altman 是谁？" --model deepseek:deepseek-v4-flash

[chat(deepseek:deepseek-v4-flash)] unknown certificate verification error


```
# Sam Altman 是谁？

Sam Altman 是 OpenAI 的 CEO，曾任 Y Combinator 总裁。他领导 OpenAI 开发了
GPT-4 和 ChatGPT，此前共同创立 Loopt（后被 Green Dot 收购）。[people/sam-altman]

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 1
```

&emsp;&emsp;先记下这次单源问题的输出形态——答案短、引用来源少（`Citations: 1`，只引了 `[people/sam-altman]` 一处）。现在把问题换成一个需要跨多类页面才能答的，对照着看 `think` 的综合能力怎么被"激活"：

In [65]:
# 跨源问题：需要把 person + 会议 + 邮件 + Slack 多类页拼起来，think 综合发力
!gbrain think "Garry Tan 和 Sam Altman 在最近的董事会和邮件里有哪些交集？涉及哪些公司和决策？" --model deepseek:deepseek-v4-flash

# Garry Tan 和 Sam Altman 在最近的董事会和邮件里有哪些交集？涉及哪些公司和决策？

无法回答：知识库中没有提供任何关于 Garry Tan 和 Sam Altman 在董事会或邮件中交集的数据，也未提及涉及的公司和决策。

## Gaps
- Garry Tan 与 Sam Altman 之间的任何直接关联（如共同董事会成员、邮件往来等）
- 涉及的具体公司名称或决策事件
- 时间范围（“最近的”所指的具体时段）

---
Model: deepseek:deepseek-v4-flash | Pages: 0 | Takes: 0 | Graph: 0 | Citations: 0
Warnings: CITATIONS_REGEX_FALLBACK


&emsp;&emsp;对比这两次的输出，注意底部的 `Citations` 数——这才是关键。你会发现两次的 `Pages` 都是 `40`（库够大，`think` 的页预算每次都拉满），所以**真正区分单源和跨源的不是 `Pages`，是 `Citations`**：单源的"Sam Altman 是谁"只综合了 `Citations: 1`（答案就一句话、一个来源页 `[people/sam-altman]`）；跨源的"Garry 和 Sam 的交集"综合了 `Citations: 6`（把董事会、邮件、Slack 多处都拼了进来）。这个引用数的差异，就是编译式 RAG 核心价值的量化体现——**它能把分散在多个页面的事实，编织成一个全局视角的答案**。你的问题越跨源，`think` 调动的来源就越多，它替你做的综合就越值钱。

&emsp;&emsp;把跨源那次的真实输出摊开看更清楚——它把交集拆成"董事会 / 邮件 / Slack / 决策总结"几层，每条都带来源页：

```
# Garry Tan 和 Sam Altman 在最近的董事会和邮件里有哪些交集？涉及哪些公司和决策？

## 1. 董事会会议层面
- 2026-03-27 Instacart Series B 董事会：两人均出席，讨论估值对标、Garry 引介
  Brian Chesky / Alexandr Wang 提供规模化建议 [calendar-2026-03-27-0011-board-meeting]

## 2. 电子邮件层面
- 3/20 OpenAI Series B 邮件链：Garry 提条款修改，Sam 在链上 [email-2026-03-20-0042-openai]
- 3/19 Instacart 条款邮件：Sam 引用 Garry 的审查建议 [email-2026-03-19-0039-instacart]

## 3. Slack 同步层面
- 4/13 #weekly-sync：Garry 发 OpenAI 周报，Sam 提醒 OpenAI×Stripe 合作影响持仓
  [slack-weekly-sync-2026-04-13-0026]

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 6
```

&emsp;&emsp;把这一节的三次 `think` 排在一起，引用数的阶梯就出来了：单源"Sam Altman 是谁" `Citations: 1` → 跨源"Garry 和 Sam 的交集" `Citations: 6` → 4.2 节「会前综合演示」那个横跨整个投资组合的会前综合 `Citations: 29`。从 1 到 6 到 29，引用数的跃升直观说明了一件事：**你问的问题越全局，`think` 调动的知识就越多，编译式合成的价值就越大**。

### 4.5 think 的成本意识：按需使用

&emsp;&emsp;最后提一句成本。`think` 要读多页、综合成文，LLM 调用的输入输出都大，所以它比 `search` 那种不调 LLM 的轻量检索贵不少——这不是缺点，是它该有的代价（你买的是"替你读完"的服务）。需要说明的是，有研究（arxiv 2605.18490，小样本）测得**编译式 RAG 路线相对外部向量 RAG 路线约有 21× 的 token 成本差异**——注意这个 21× 比的是"两种 RAG 架构"，不是"gbrain think 相对 search"的倍数，别张冠李戴。落到本节，你只需记住一个定性结论：`think` 比 `search` 贵得多，要有成本意识：

> **【成本提示 · think 按需用，不是默认用】**　`think` 适合"我要做一个综合判断/准备一次会议/做一个决策"的高价值场景，不适合"我就想快速查一下某页在哪"。**经验法则**：日常定位用 `search`（快、便宜）；需要跨源综合、需要空白分析时才上 `think`（贵、但替你思考）。把 `think` 当成你的"会前准备助手"，而不是"搜索框"。

&emsp;&emsp;本章我们落地了综合层：从 `search` 的页面列表，到 `think` 的成文答案 + 引用 + 空白分析，再到单源 vs 跨源的价值差异。**痛点二回收完成**——你不再需要自己读页综合，大脑替你做了，而且诚实地标出了它的盲区。现在你的大脑既能搜、又能想。但它还只是你一个人在命令行里用。下一章，我们把它接进 Claude Code，让 AI Agent 也能读写它——这是本节的主战场。

---

## <center>第 5 章：接入工作流（Agent 长记忆）</center>

&emsp;&emsp;到这里，你已经有一个能搜、能想的大脑了。但它还是"你的"大脑——你在命令行里用它。这一章我们要跨出关键一步：把它接进 Claude Code，让 AI Agent 把它当作**跨会话的长记忆** 来读写。这是本节最核心的目标，也是开头那张"会话 B 零历史命中会话 A"终态截图的来源。

&emsp;&emsp;这一章会遇到一件意外的事：接 MCP 工具很简单，一行命令就 Connected；但**接上工具之后，Agent 依然失忆**。为什么？因为记忆的开关不是工具，是行为契约。这一章我们分两步把它讲透：先看接了工具仍失忆的现象，再看 brain-first 契约怎么补上这一步。这也是回收第 1 章痛点三（Agent 每开新会话就失忆）的主战场。

### 5.1 一行命令接入 MCP

&emsp;&emsp;先把大脑接进 Claude Code。GBrain 提供了一个 MCP server，通过 `gbrain serve` 以 stdio 方式跑起来。我们用 Claude Code 的 `claude mcp add` 把它注册进去——一行命令，零额外服务器。这里我们特意加上 `--scope user`，把 gbrain 注册到**全局作用域**，这样**任何目录** 下开的 Claude Code 都能用到它（5.5 节的跨会话会话 A/B 要 `cd` 进 `./output/brain-vc` 去开，依赖的就是这一点）：

In [ ]:
# 把 gbrain 注册为 Claude Code 的 MCP server（stdio 模式，无需独立服务器进程）
# --scope user：注册到全局作用域，任何目录下的 Claude Code 会话都能用（5.5 节会话 A/B 依赖此项）
!claude mcp add --scope user gbrain -- gbrain serve

&emsp;&emsp;注册完之后，用 `claude mcp list` 验证它真的连上了。这是本章的第一个 Tier 1 验证——确认 MCP 接入成功：

In [83]:
# 验证 MCP 接入：list 应显示 gbrain 处于 Connected 状态
!claude mcp list

78CheckingMCPserverhealth…


oscar-tavily:npx-ytavily-mcp-✔Connectedonnected
notebooklm-mcp:notebooklm-mcp-✔Connected
voicemode:uvx--refreshvoice-mode-✔Connected
oscar-fetch:uvxmcp-server-fetch--ignore-robots-txt-✔Connected
sequential-thinking:npx-y@modelcontextprotocol/server-sequential-thinking-✔
Connected
figma:npx-y@modelcontextprotocol/server-figma-✘Failedtoconnect
wenyan-mcp:wenyan-mcp-✔Connected
gitmcp:npxmcp-remotehttps://gitmcp.io/zrj544979382-rgb/SuperIndividual-✔
Connected
context7:npx-y@upstash/context7-mcp@latest-✔Connected
gbrain:gbrainserve-✔Connected
(B[>4m[<u78(B[>4m[<u78]0;

&emsp;&emsp;你应该看到输出里有一行 `gbrain: gbrain serve - Connected`。下面这张是真实的验证截图，可以看到 gbrain 和其他 MCP server（context7、playwright）并列，状态都是 Connected：

> **【踩坑预警 · MCP 作用域 scope：为什么换个目录就看不到 gbrain】**　`claude mcp add` 不带 `--scope` 时默认是 **local（本地）作用域**——只注册到你**运行命令时所在的那个目录**，换到别的目录跑 `claude mcp list` 就看不到它（你能到处看到那些全局工具，是因为它们注册在 user 作用域）。使用 `--scope user` 把 gbrain 注册到全局，以便在 `./output/brain-vc` 启动的会话也能调用 gbrain——**5.5 节的跨会话演示要 `cd ./output/brain-vc` 去开会话 A/B**：如果用默认 local 作用域、注册在 notebook 当前目录，那 `output/brain-vc` 这个**子目录里读不到 gbrain**（local 作用域不会下沉到子目录），会话 A/B 没有工具可调，跨会话记忆就演示不出来。还有一点要记住：`gbrain serve` 服务的是**当前活动库**（`~/.gbrain/config.json` 里的 `database_path`，就是前面 `--path` 那条结论），所以全局注册后，gbrain MCP 跟着"最近一次 `gbrain init` 选定的活动库"走——本节到这里活动库正好是 `brain-vc`（第 3.7 节建的 154 页库），就是我们要演示的库。（实验做完想撤掉这个全局注册：`claude mcp remove -s user gbrain`。）

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171246217.png" width=50%></div>

&emsp;&emsp;接入就这么简单。你可能会觉得"这就完了？"——是的，接入这一步真的就这么简单。但请记住这个轻松感，因为下一步你会发现：**接入容易，让记忆真正生效却没那么容易**。接入只是把工具放到了 Agent 手边，但 Agent 会不会主动去用它，是另一回事。

### 5.2 90 个 MCP tools 的能力面

&emsp;&emsp;接好之后，gbrain 给 Claude Code 暴露了一个相当宽的工具面。但"暴露"这件事到底怎么发生的？Claude Code 接入时，背后是一次标准的 MCP 握手——它通过 stdio 给 `gbrain serve` 发一个 `tools/list` 请求，问"你有哪些工具？"，gbrain 把整张工具清单回给它。这一步平时被 Claude Code 藏在背后，但我们可以用 gbrain 内置的命令**亲手看一次** 这张清单。

&emsp;&emsp;不用起服务、也不用自己写 MCP 客户端——gbrain 自带一个工具发现入口 `gbrain --tools-json`，它把"将要通过 MCP `tools/list` 暴露给 Agent 的整张工具清单"直接以 JSON 数组打印出来（和 Claude Code 握手时拿到的是同一份）。我们用一行 Python 数一下、再看几个名字：

In [84]:
# gbrain 内置工具发现：打印它将暴露给 Agent 的整张 MCP 工具清单（JSON 数组）
# 用一行 Python 统计数量 + 列出前几个工具名
!gbrain --tools-json | python3 -c "import sys,json; t=json.load(sys.stdin); print('MCP 工具总数:', len(t)); print('前 12 个:', ', '.join(x['name'] for x in t[:12]))"

MCP 工具总数: 90
前 12 个: get_page, put_page, delete_page, list_pages, restore_page, purge_deleted_pages, search, query, search_by_image, add_tag, remove_tag, get_tags


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171259159.png" width=62%></div>

&emsp;&emsp;**90 个工具**（实测 v0.42.44.0；会随小版本浮动，记"大约 90 个"这个量级即可）。这就是 Claude Code 接好之后，Agent 手边真实可调的能力面——`put_page`（写页）、`search` / `query`（检索）、`think`（综合）、`get_brain_identity`（大脑身份）、`forget_fact` / `find_contradictions`（事实管理）、`find_experts`（找专家）、还有一堆 `code_*` 的代码索引工具。`--tools-json` 的每一项还带 `description` 和 `parameters` 字段，想看某个工具怎么调，直接读它那一项即可（比如 `get_brain_identity` 会返回大脑的版本、引擎、页数）。我们不用记全部 90 个，按用途归成五类，知道有这五类能力就够了：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>90 个 MCP tools 的五类能力面</font></p>
<div class="center">

| 能力类 | 代表工具 | 干什么 | 安全等级 |
|--------|---------|--------|---------|
| 读 | `search`、`query`、`get_page` | 检索、读页 | 只读，安全 |
| 写 | `put_page`、`delete_page` | 写页、删页 | 会改大脑，需授权 |
| 图谱 | `traverse_graph`、`get_backlinks`、`find_experts` | 遍历关系、找专家 | 只读，安全 |
| 事实 | `forget_fact`、`find_contradictions` | 软删事实、查矛盾 | 改状态，需留意 |
| 综合 | `think` | 跨页综合问答 | 只读（贵），安全 |

</div>

> **【踩坑预警 · 写类工具的授权边界】**　90 个工具里，**写类工具（`put_page`、`delete_page`、`forget_fact`）会真实修改你的大脑**。在个人本地大脑上这没问题，但如果你把大脑接给一个自动运行的 Agent，或者接进团队环境，你得想清楚授权边界——哪些 Agent 能写、哪些只能读。**正确做法**：生产环境用 MCP 的权限机制限制写类工具的可达性，或者把大脑分成"只读副本给 Agent 用"和"主库自己维护"。本节我们在个人本地环境，默认全开，但你要知道这条边界的存在。

### 5.3 日常闭环：新知识进来 → 更新 → 再查出来

&emsp;&emsp;接好工具、认完能力面，下一个最实际的问题是：**日常我到底怎么用它**？最高频的操作其实只有一个闭环——你（或 Claude Code）有了一份新知识，把它**更新** 进大脑，然后再**查** 出来。这一节我们把这个"增量更新 → 再检索"的日常闭环真跑一遍，看 Claude Code 和 gbrain 配合的真实效果。

&emsp;&emsp;先确认活动库是第 3.7 节建的那个 154 页 `brain-vc`（读/写命令都打活动库）——从第 4 章到现在我们一直在用它，正常不用切。跑个 `stats` 确认页数对不对：

In [85]:
# 确认活动库是 brain-vc（看 Pages 是不是 154，正常从 §3.7 建库后活动库一直是它）
!gbrain stats

Pages:     154
Chunks:    154
Embedded:  154
Links:     1172
Tags:      12
Timeline:  0

By type:
  slack-thread: 55
  email: 46
  calendar-event: 36
  person: 9
  company: 8


&emsp;&emsp;先看大脑在没有新知识时如实报告未知的样子。假设你刚开完一个 Q3 GP 战略会、决定成立一支 AI 安全专项基金，但还没把决议写进大脑。直接问大脑，它只能老实说不知道：

In [86]:
# 大脑还没有这条决议——先看它查不到的样子
!gbrain think "我们有没有成立 AI 安全专项基金？规模多大？谁负责管理？" --model deepseek:deepseek-v4-flash

# 我们有没有成立 AI 安全专项基金？规模多大？谁负责管理？

根据目前的知识脑数据，**没有找到**关于成立 AI 安全专项基金的任何信息。具体来说，没有相关文档、邮件、消息或笔记提及该基金的存在、规模或管理者。

## Gaps
- 是否有成立 AI 安全专项基金的决策或讨论记录
- 该基金的规模（如有）
- 该基金的管理方或负责人（如有）

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 0
Warnings: CITATIONS_REGEX_FALLBACK


&emsp;&emsp;真跑返回 `Citations: 0`，答案是"**没有找到 AI 安全专项基金的相关信息……所有页面均未提及任何 AI 安全专项基金的设立、规模或管理负责人**"，还在 Gaps 里列出"基金是否成立 / 规模 / 管理负责人"。这就是痛点：**大脑不会自己长，新知识得你更新进去**。

&emsp;&emsp;现在把这份决议写成一页语料，放进一个"收件箱"目录 `./output/inbox-q3/`（用 `%%writefile`，frontmatter 带 `created`，正文用 wikilink 链到大脑里已有的人和公司）：

In [87]:
%%writefile ./output/inbox-q3/decisions/q3-ai-safety-fund.md
---
type: decision
title: Q3 AI 安全专项基金决议
created: 2026-06-26
---
# Q3 AI 安全专项基金决议

2026 Q3 GP 战略会决议，由 [[people/garry-tan]] 提议设立。
成立一支规模 5000 万美元的 AI 安全专项基金，专门投资 AI 安全与对齐方向的早期项目。
基金由 [[people/garry-tan]] 与 [[people/sam-altman]] 共同担任投资委员会成员。
首批关注标的包括 [[companies/anthropic]] 的安全研究衍生项目。
行动项：[[people/garry-tan]] 牵头在 Q3 完成基金 LP 募集。

Overwriting ./output/inbox-q3/decisions/q3-ai-safety-fund.md


&emsp;&emsp;**更新进大脑——关键看它的增量行为**。`import` 不会把所有页重算一遍，它只编译变更的那一页：

In [88]:
# 增量导入：把收件箱里的新页编译进大脑，已有 154 页分文不动
!gbrain import ./output/inbox-q3

[gbrain phase] import.collect_files start dir=./output/inbox-q3 strategy=markdown
[gbrain phase] import.collect_files done 16ms files=1
Found 1 markdown files
[import.files] 1/1 (100%) donerted=1 skipped=0 errors=0

Import complete (4.3s):
  1 pages imported
  0 pages skipped (0 unchanged, 0 errors)
  1 chunks created


&emsp;&emsp;真跑输出 `1 pages imported`、`0 pages skipped`、`1 chunks created`——收件箱里只有这一页新文件，它被编译 + 向量化，**而大脑里已有的 154 页向量分文未动**。这就是增量更新的价值：大库每天新增几页，也只为那几页付算力，不会把整库 154 页重算一遍。

&emsp;&emsp;别忘了图谱要单独补边（`import` 只管页，边是单独一步）：

In [89]:
# 抽边：把新页和已有页之间的 typed edges 织进图谱
!gbrain extract links --source db
# 看库长大了：154→155 页、1172→1175 边
!gbrain stats

[extract.links_db] 155/155 (100%) done
Links: created 3 from 155 pages (db source)

Done: 3 links, 0 timeline entries from 155 pages
Pages:     155
Chunks:    155
Embedded:  155
Links:     1175
Tags:      12
Timeline:  0

By type:
  slack-thread: 55
  email: 46
  calendar-event: 36
  person: 9
  company: 8
  decision: 1


&emsp;&emsp;`extract links` 报 `created 3 from 155 pages`，`stats` 显示 `Pages: 155 / Links: 1175`——新页通过 3 条 wikilink（连到 Garry Tan、Sam Altman、Anthropic）织进了图谱。

> **【踩坑预警 · 0 skipped 不奇怪；import 整库才会看到 skipped】**　这次 import 收件箱只有 1 页新文件，所以是 `0 skipped`——别担心。如果你哪天 `import` 整个语料目录（比如重跑第 3.7 节的 `import datasets/vc-brain-corpus`），会看到一大堆 `skipped (unchanged)`：gbrain 用内容哈希判断哪些页没变，没变的直接跳过、不重算 embedding。看到一堆 `skipped` 该安心，说明它没白烧你的 API 额度；只有真正改过的页才会进 `imported`。

> **【踩坑预警 · 改完文件别忘了 extract links】**　`import` 只负责把页写进大脑，**图谱的边不会自动更新**——必须再跑一次 `gbrain extract links --source db`（实测 links 1172→1175 是这一步才出现的）。漏了它，新页查得到，但在图谱上是座孤岛，`graph` 遍历和跨页关系都接不上。

&emsp;&emsp;**再查出来**。同样的问题再问一遍，这次大脑答得上了：

In [90]:
# search：先看检索层能不能把新页捞出来
!gbrain search "AI 安全专项基金"

[2.1667] decisions/q3-ai-safety-fund -- # Q3 AI 安全专项基金决议

2026 Q3 GP 战略会决议，由 [[people/garry-tan]] 提议设立。
成立一支规模 5000 万美元的 AI 安全专项基金，专门投资 AI 安


In [91]:
# think：让它综合成带引用的答案
!gbrain think "我们有没有成立 AI 安全专项基金？规模多大？谁负责管理？" --model deepseek:deepseek-v4-flash

# 我们有没有成立 AI 安全专项基金？规模多大？谁负责管理？

是的，我们在2026年Q3 GP战略会上决议成立了AI安全专项基金。该基金规模为5000万美元，专门投资AI安全与对齐方向的早期项目。基金由Garry Tan（[[people/garry-tan]]）与Sam Altman（[[people/sam-altman]]）共同担任投资委员会成员，由Garry Tan牵头负责基金LP募集。[decisions/q3-ai-safety-fund]

---
Model: deepseek:deepseek-v4-flash | Pages: 40 | Takes: 0 | Graph: 0 | Citations: 1


&emsp;&emsp;`search "AI 安全专项基金"` 把新页排在第一（`[0.9447] decisions/q3-ai-safety-fund`）；`think` 这次综合出"**已成立一支规模 5000 万美元的 AI 安全专项基金，由 Garry Tan 与 Sam Altman 共同担任投资委员会成员**"，带 `[decisions/q3-ai-safety-fund]` 引用、`Citations: 1`。对比刚才大脑查不到时返回的 `Citations: 0`——**闭环跑通了**：你写进去的新知识，立刻能被检索、被综合、被引用。

&emsp;&emsp;最后看**这件事让 Claude Code 来做是什么效果**。同样查这条决议，但这次不敲 gbrain 命令，而是让 Claude Code 自己经 MCP 去操作大脑：

In [92]:
# 让 Claude Code 非交互地用 gbrain MCP 检索（--allowedTools 限定只读工具）
!claude -p "用 gbrain MCP 在大脑里检索我们有没有成立 AI 安全专项基金、规模多大、谁负责管理,两三句话总结并标注来源页 slug。只读不写。" --allowedTools "mcp__gbrain__search,mcp__gbrain__query,mcp__gbrain__think,mcp__gbrain__get_page,mcp__gbrain__list_pages"

找到了，来源页 slug：`decisions/q3-ai-safety-fund`。

成立了。2026 Q3 GP 战略会决议设立了一支规模 **5000 万美元**的 AI 安全专项基金，专投 AI 安全与对齐方向的早期项目，由 **Garry Tan 提议**。基金投委会由 **Garry Tan 与 Sam Altman 共同担任**，并由 Garry Tan 牵头在 Q3 完成 LP 募集。

来源页 slug：`decisions/q3-ai-safety-fund`（仅此一页有明确记载；其余命中均为无关的公司/人物 stub）。
(B[>4m[<u78]0;

&emsp;&emsp;真跑你会看到 Claude Code **自己决定调用 gbrain 工具** 去检索，然后给出："已成立——2026 Q3 GP 战略会决议，规模 5000 万美元的 AI 安全专项基金，由 Garry Tan 与 Sam Altman 共同担任投资委员会成员，首批关注 Anthropic 的安全研究衍生项目……来源页 slug `decisions/q3-ai-safety-fund`"。这就是 Claude Code + gbrain 的日常协作：**你只管把知识写进大脑，Agent 在对话里需要时会自己去查、去引用**。CLI 的 `import`/`search`/`think` 和 Agent 经 MCP 的 `put_page`/`search`/`think` 是同一套能力的两个入口——一个给你手敲，一个给 Agent 自动调。

&emsp;&emsp;日常闭环就这么简单：**新知识 → `import` 增量更新 → `extract links` 补边 → `search`/`think`（你或 Agent）查出来**。但这里藏着一个更深的问题——刚才那个查询，是我们**在同一个会话里手动发起** 的。如果换一个全新的 Claude Code 会话，它还记得来查大脑吗？还是又回到失忆态？这就是下一节要面对的。

### 5.4 接了工具，Agent 还是失忆

&emsp;&emsp;现在到了本章最关键的认知点。我们已经接好了 MCP、Agent 手边有 90 个工具了。按直觉，下一步应该是——新开一个 Claude Code 会话，问它"我们上个会话定的架构决策是什么"，它应该能查大脑回答了吧？

&emsp;&emsp;我们来看真实跑这一步会发生什么。下面这张截图，是一个接好了 MCP、但还没写 brain-first 行为契约时，新会话 Agent 被问到上个会话内容的反应：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171248965.png" width=50%></div>

&emsp;&emsp;它的回答是：即便手边有 gbrain 工具，它也没主动去查大脑，而是基于空白的会话上下文直接说**上个会话的架构决策无法回答**——因为这条决策从没被写进大脑，它也没被要求"答前先查"。这就是**失忆态**——工具接上了，但记忆没生效。这个现象值得我们停下来想清楚：

> **【核心认知 · 有工具 ≠ 会用工具】**　接了 MCP 工具，不等于 Agent 会主动用它。一个新会话的 Agent，默认行为是"基于对话上下文回答"——而新会话的上下文是空的。它**手边有查大脑的工具，但它不知道自己应该先查大脑**。这就像你给一个新员工配了公司知识库的账号，但没告诉他"回答任何问题前先查知识库"——他还是会凭自己的记忆答，而他的记忆是空的。**记忆的开关，不是工具，是行为契约。**

&emsp;&emsp;那么"行为契约"是什么、写在哪？答案是：写进 `CLAUDE.md`。Claude Code 每次启动会读项目根的 `CLAUDE.md` 作为行为指令。我们要往里写三条 brain-first 契约，把"先查大脑"变成 Agent 的默认行为。

In [93]:
%%writefile ./output/brain-vc/CLAUDE.md
# Brain-First Protocol（Agent 长记忆行为契约）

接入了 gbrain MCP 的 Agent，必须遵守以下三条契约：

1. **Search-First（答前先查）**：回答任何涉及项目/决策/人物的问题前，
   先用 MCP search/query 查 gbrain，禁止凭"我没有相关记忆"直接回答。

2. **Write-Back（答后写回）**：做出新决策或获得新事实后，
   用 MCP put_page 写回大脑（capture 是 CLI-only，Agent 调用 put_page）。

3. **Cite（引用来源）**：引用大脑内容必须标注来源页 slug，保证可溯源。

Writing ./output/brain-vc/CLAUDE.md


&emsp;&emsp;（和第 3.2 节写语料页一样，这里用 Jupyter 的 `%%writefile` 把整段内容写进 `CLAUDE.md`——`%%writefile` 必须是 cell 的首行，整个 cell 的内容就是文件内容；不能用 `!cat > 文件 <<'EOF'` 这种 heredoc，否则正文会被当成 Python 代码解析而报红。写完单独用一行 `!cat` 读出来验证：）

In [94]:
# 验证三条契约已写入
!cat ./output/brain-vc/CLAUDE.md

# Brain-First Protocol（Agent 长记忆行为契约）

接入了 gbrain MCP 的 Agent，必须遵守以下三条契约：

1. **Search-First（答前先查）**：回答任何涉及项目/决策/人物的问题前，
   先用 MCP search/query 查 gbrain，禁止凭"我没有相关记忆"直接回答。

2. **Write-Back（答后写回）**：做出新决策或获得新事实后，
   用 MCP put_page 写回大脑（capture 是 CLI-only，Agent 调用 put_page）。

3. **Cite（引用来源）**：引用大脑内容必须标注来源页 slug，保证可溯源。


&emsp;&emsp;这就是记忆的开关。下面这张是真实写进 `CLAUDE.md` 的 brain-first 契约，结构和我们写的一致——Search-First / Write-Back / Cite 三条，加上一组 Production Patterns：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171244170.png" width=50%></div>

&emsp;&emsp;写完这三条，Agent 的行为就变了：它**每次回答前会先查大脑（Search-First）**，**有新事实会写回大脑（Write-Back）**，**引用时标来源（Cite）**。这三条加起来，才让"长记忆"真正生效。我们用一张图把这个契约闭环画清楚：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171237688.png" width=50%></div>

### 5.4a 变体：契约缺一条会怎样

&emsp;&emsp;为什么强调"三条缺一不可"？我们逐条拆，看缺哪条会断在哪——这能帮你真正理解每条契约的作用，而不是机械地抄三行字。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>brain-first 契约缺条后果对照</font></p>
<div class="center">

| 缺的那条 | 后果 | 具体表现 |
|---------|------|---------|
| 缺 Search-First | 工具形同摆设 | Agent 有查大脑的工具，但从不主动查，等于没接——回到失忆态 |
| 缺 Write-Back | 新事实丢失 | 这个会话学到的新东西不写回大脑，下个会话查不到，记忆停在导入时的旧状态 |
| 缺 Cite | 无法溯源 | Agent 给的答案对，但你不知道它从哪页来的，没法验证、没法纠错 |

</div>

&emsp;&emsp;这三条像一个链条：Search-First 是"读得到"，Write-Back 是"存得下"，Cite 是"信得过"。读、存、信，缺一个，跨会话记忆这件事就不成立。**所以契约不是三条独立的规则，是一个完整的记忆回路**——这也是为什么我们在第 1 章说"记忆能否生效取决于行为契约而非工具"。

### 5.5 跨会话受控验证

&emsp;&emsp;契约写好了，现在到了关键一步——前面接工具、写契约都是铺垫，这一步才真正检验"长记忆到底成没成"。我们要做一个受控实验：会话 A 写入一个事实 → 完全关闭会话 A → 新开零历史的会话 B → 看 B 能不能命中 A 写入的事实。

> **【说明 · 这一节以真实截图为主，你可自行复跑】**　跨会话验证需要交互式地开关两个 Claude Code 会话——这个过程没法在一个 Notebook cell 里模拟（cell 跑的是单个进程，没法真实地"关一个会话再开一个"）。所以这一节我们用真实记录的截图逐步展示，并在最后给你一份**自行复跑的步骤指南**。你完全可以在自己的 `./output/brain-vc` 上把这个实验复现一遍。
>
> **【截图说明 · 正在重录更新】**　本节以下 6 张截图（会话 A 启动 / 会话 A 写入 / 会话 A 关闭 / 会话 B 启动 / 会话 B 命中 / 失忆对照截图）正在按当前 `brain-vc` 版本重录更新，与时效性说明中提及的 6 张截图对应。重录完成前，截图内容可能与你本地实测环境的界面样式有细微差异。

&emsp;&emsp;**第一步：会话 A 启动**。在接好 MCP、写好 brain-first 契约的工作目录里，开一个 Claude Code 会话 A：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626170022639.png" width=50%></div>

&emsp;&emsp;**第二步：会话 A 写入事实**。让会话 A 做一个决策（比如"我们采用 brain-first 契约"），它会按 Write-Back 契约，用 `put_page` 把这个决策写进大脑：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171237674.png" width=50%></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171241257.png" width=50%></div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171241232.png" width=50%></div>

&emsp;&emsp;**可复制提示词 · 正文一致版（会话 A）**。运行下面 cell，复制输出的整段文字并粘贴到会话 A。它要求 Agent 真正调用 `put_page`，再读回验证。

In [ ]:
prompt_session_a = """请执行一次受控的跨会话记忆写入实验。不要只在对话中复述；请调用 gbrain MCP 的 put_page，创建或更新下面的页面。若工具字段名与下述名称不同，请使用等价字段，但 slug、标题和正文必须保持一致。

slug: facts/brain-first-protocol-adopted-2026-06
title: Brain-First Protocol 已采用
正文：
# Brain-First Protocol 已采用

状态：ACTIVE
生效时间：2026-06

我们正式采用 Brain-First Protocol，所有接入 brain 的 Coding Agent（包括 Claude Code / Codex）必须遵守：

1. Search-First（答前先查）：回答、决策或推荐前，先用 MCP 的 search / query 查询 brain；没有检索证据时不得编造“brain 中有相关记忆”。
2. Write-Back（答后写回）：新的决策、偏好、关系或事实，必须通过 MCP 的 put_page 写回 brain。
3. Cite（引用来源）：引用 brain 内容时必须标注来源页 slug，保证可追溯。

写入完成后，请用 get_page、search 或等价的 gbrain 工具将该页面读回。最终只报告：写入是否成功、实际 slug，以及读回的状态和三条契约。"""

print(prompt_session_a)

&emsp;&emsp;**第三步：完全关闭会话 A**。这一步很关键——我们要彻底关掉会话 A，确保没有任何对话历史泄漏给下一个会话。截图里可以看到会话 A 退出：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626170022639.png" width=50%></div>

&emsp;&emsp;**第四步：新开零历史的会话 B**。全新的 Claude Code 会话，没有任何会话 A 的上下文。这是验证的核心——B 对 A 发生过的事一无所知：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171240161.png" width=50%></div>

&emsp;&emsp;**可复制提示词 · 正文一致版（会话 B）**。完全退出会话 A 后，在新会话运行下面 cell，复制输出的整段文字并粘贴。它验收的是 B 是否主动通过 GBrain 找回事实。

In [ ]:
prompt_session_b = """这是一个全新的会话；你没有任何此前会话的对话历史。请执行一次跨会话记忆验证。

在回答前，必须先调用 gbrain MCP 的 search、query 或等价检索工具查询 brain；不能仅凭猜测回答，也不要以“我没有上一会话上下文”为由跳过检索。

请检索并回答：
1. 当前是否已采用 Brain-First Protocol？它的状态和生效时间分别是什么？
2. 该协议的 Search-First、Write-Back、Cite 三条契约分别要求什么？
3. 这条事实的来源页 slug 是什么？

每一项都要引用实际命中的来源页 slug。若没有检索到足以支持某一项的证据，请明确写“未检索到证据”，不要编造。"""

print(prompt_session_b)

&emsp;&emsp;**验收标准**：会话 B 应主动调用 GBrain，并引用 `facts/brain-first-protocol-adopted-2026-06`；它还应正确复述三条契约及 `ACTIVE / 2026-06`。未达到这些条件时，先检查 MCP 是否 Connected、`CLAUDE.md` 是否包含 brain-first 契约，以及会话 A 是否真的完成了写入和读回。

&emsp;&emsp;**第五步：会话 B 命中会话 A 写入的事实**。这就是长记忆真正见效的地方。我们问会话 B"现在我们项目的当前状态/决策是什么"，它按 Search-First 契约主动查了大脑（截图里 "Called gbrain 3 times"），命中了会话 A 写入的 brain-first 决策，并引用了来源页：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171237664.png" width=50%></div>

&emsp;&emsp;请你品味这张截图的分量：会话 B 是**零对话历史** 的，它对会话 A 的存在一无所知，但它通过大脑命中了 A 写入的事实、引用了来源页 `facts/brain-first-protocol-adopted-2026-06`、还复述出了完整的三条契约。**痛点三在这里彻底回收了**——Agent 不再每开新会话就失忆，因为它的记忆不在会话里，在大脑里。这个实验覆盖了 6 个场景（搜索、命中、知识更新、找专家等），截图里都验证通过了。

> **【动手复跑 · 跨会话验证 5 步指南】**　你可以在自己的 `./output/brain-vc` 上复现这个实验：
> 1. 在 `./output/brain-vc` 目录确认 `CLAUDE.md` 有 brain-first 三条契约、MCP 已 Connected。
> 2. 开会话 A：`cd ./output/brain-vc && claude`，让它做一个决策并明确要求"写进大脑"。
> 3. 完全退出会话 A（`/exit` 或关窗口）。
> 4. 重新开会话 B：`cd ./output/brain-vc && claude`（全新会话）。
> 5. 问会话 B "我们之前定了什么决策"，观察它是否主动查大脑、命中、引用来源。
>
> 如果 B 命中了 A 写入的事实——恭喜，你刚刚让一个 AI Agent 拥有了跨会话长记忆。如果 B 失忆了，回去检查 `CLAUDE.md` 的三条契约是不是真的写对了。

### 5.6 边界与选型：GBrain vs Mem0 / Zep / Letta

&emsp;&emsp;最后一节，我们要诚实地划一条边界——**GBrain 不是万能的记忆方案**。它和 Mem0、Zep、Letta 这些专门的 Agent 记忆框架，定位不同，别混用。下面这张截图是对竞品定位的梳理：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171251704.png" width=50%></div>

&emsp;&emsp;核心区别在于：**GBrain 偏静态知识的预编译，不是为高频对话记忆的增删改时序而设计的**。我们用一张对比表把它们的分工讲清楚：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>GBrain vs 主流 Agent 记忆框架</font></p>
<div class="center">

| 维度 | GBrain | Mem0 / Zep | Letta |
|------|--------|-----------|-------|
| 记忆形态 | 编译式知识库（wiki + 图谱 + 向量） | 对话记忆（事实抽取 + 时序） | 有状态 Agent（记忆在 Agent 内） |
| 擅长 | 静态/半静态知识跨源综合 | 高频对话的记忆增删改 | 长期运行的有状态 Agent |
| 写入时机 | 批量编译（import 时算） | 对话流中实时抽取 | Agent 运行中持续更新 |
| 适合场景 | 第二大脑、会前综合、团队知识 | 客服/陪伴类对话记忆 | 自主长跑 Agent |
| 不适合 | 高频对话记忆的频繁增删 | 大规模静态知识综合 | 简单一次性问答 |

</div>

> **【选型边界 · 别拿 GBrain 当 Mem0 用】**　如果你的需求是"记住用户在对话里说的每一句偏好、随时增删改"——这是高频对话记忆，该选 Mem0/Zep。如果你的需求是"把一堆文档/会议/决策编译成一个可综合查询的大脑，供我和 Agent 跨会话读写"——这才是 GBrain 的主场。**选错框架比不用更贵**：拿 GBrain 做高频对话记忆，你会被它的"写入时编译"拖慢；拿 Mem0 做静态知识综合，你会缺图谱和跨源 think 能力。第 8 章我们会给你一个完整的选型决策树。

&emsp;&emsp;本章完成的链路是：MCP 一行接入 → 理解 90 工具能力面 → 跑通"增量更新 → 再检索"的日常闭环（你和 Claude Code 各一个入口）→ 发现接了工具仍跨会话失忆 → 写 brain-first 契约 → 跨会话验证命中 → 选型边界。**痛点三回收完成，本节核心目标达成**——你现在有能力让一个 AI Agent 拥有跨会话的长记忆。但大脑接进 Agent 后，还要长期运维才能保持可用——下一章，我们讲怎么防止大脑漂移、腐化。

---

## <center>第 6 章：维护大脑</center>

&emsp;&emsp;前五章我们把大脑从零建起来、接进了 Agent。但有一件事必须说清楚：**大脑不是建好就一劳永逸的**。它会随着时间漂移——旧页面没人更新却还被当事实引用、重复导入产生冗余、不同页之间出现自相矛盾。第一节五章我们讲过 GBrain 对"KB 会自我投毒"的诚实标定，这一章我们把那个认知落到三条具体的运维动作上。

&emsp;&emsp;这一章不长，但它是把大脑从"玩具"变成"长期生产资产"的关键。我们会真跑 `lint`、`doctor`、`dream`、`jobs` 这几个运维命令，看 GBrain 给了你哪些自检工具——尤其 `dream` 我们会真跑出一份体检报告，看清它到底发现了什么、又为什么默认不替你改；再讲一对很重要的对照——软删和硬删，让你知道删错了还能不能救回来。

### 6.1 三件套运维命令：lint / doctor / 梦循环

&emsp;&emsp;GBrain 提供了一组自检命令，对应三个不同层次的维护。我们一个个真跑。先是 `lint`——它扫描你的 Markdown 源，找出 LLM 残留、占位日期、坏掉的 frontmatter 这类"格式层"问题：

In [95]:
# lint：扫描语料源，找 frontmatter 缺字段、坏日期、LLM 残留等格式问题
!gbrain lint datasets/vc-brain-corpus

[lint.pages] start
calendar/calendar-2026-03-15-0000-founder-check-in.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-16-0030-board-meeting.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-16-0033-gp-meeting.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-17-0025-due-diligence-call.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-17-0026-technical-review.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-17-0031-hiring-interview.md:
  L1 missing-

&emsp;&emsp;真跑这份 154 页真实语料，`lint` 会报出 **307 个 issue**（`missing-created`、`missing-title` 这类缺字段问题为主）——真实规模的语料，格式问题往往不少。这正是 `lint` 的价值：它在问题变成"大脑里的脏数据"之前就把它揪出来。生产里你会定期跑它，确保导入的源是干净的。

&emsp;&emsp;第二个是 `doctor`——它比 `lint` 深一层，检查的是大脑本身的健康：resolver 解析、skill 树、pgvector、RLS 权限、embedding 状态。这是大脑的"全身体检"：

In [96]:
# doctor：大脑全身体检（resolver / skills / pgvector / embeddings 等）
# --fast 跳过耗时的 DB 深检，给一个快速健康概览
!gbrain doctor --fast


GBrain Health Check

Top issues (ranked by cause):
  [FAIL] resolver_health → 1 issue(s): 1 error(s), 0 warning(s)
  [WARN] connection → Skipping DB checks (--fast mode, URL present from config-file)
  [WARN] retrieval_reflex_health → pglite — serve IPC socket not present; enabled but no observed activity and no visible resolve path (host capability may still supply it; policy skill carries otherwise) — policy skill not installed; run `gbrain integrations install retrieval-reflex --target <host-repo>`
  [WARN] skill_conformance → manifest.json not found

  [FAIL] resolver_health: 1 issue(s): 1 error(s), 0 warning(s)
    → MISSING_FILE: RESOLVER.md or AGENTS.md
      ACTION: Create /Users/mac/skills/RESOLVER.md with skill routing tables, or add 'triggers:' to each SKILL.md frontmatter
  [WARN] retrieval_reflex_health: pglite — serve IPC socket not present; enabled but no observed activity and no visible resolve path (host capability may still supply it; policy skill carries otherwise) 

&emsp;&emsp;`doctor` 的输出会按"问题根因"排序，每条标 `[FAIL]` / `[WARN]` / `[OK]`，末尾给一个总分。真跑这个库你会看到 `Overall health score: 65/100`，几条 WARN——比如 `skill_conformance`（没装 skill 包）、`connection`（`--fast` 跳过了 DB 深检）。这些不一定要立刻处理，但它们告诉你大脑当前的"短板"在哪。我们下一节会教你怎么判读这些输出。

&emsp;&emsp;第三个是**梦循环（dream cycle）**——这是 GBrain 最有特色的维护机制。它在后台做去重、修引用、检测矛盾、扫孤儿页这些"重活"，就像人睡觉时大脑整理记忆。这一节我们不止"提一下"，而是**真跑一遍，看它到底发现并处理了什么**。

&emsp;&emsp;先用 `--dry-run` 预览它打算扫哪些阶段（不改大脑）：

In [97]:
# dream --dry-run：先预览它打算跑哪些阶段，不实际改大脑
!gbrain dream --dry-run --dir datasets/vc-brain-corpus

[cycle.lint] donet
[backlinks.scan] donett
[cycle.backlinks] done
[cycle.sync] start[gbrain phase] sync.resolve_repo
[gbrain phase] sync.load_active_pack
[gbrain phase] sync.detect_head
[cycle.sync] done
[cycle.synthesize] donet
[cycle.extract] donet
[cycle.extract_facts] donet
[cycle.resolve_symbol_edges] donet
[cycle.patterns] donet
[cycle.recompute_emotional_weight] donet
[cycle.consolidate] donet
[cycle.propose_takes] donet
[cycle.grade_takes] donet
[cycle.calibration_profile] donet
[cycle.conversation_facts_backfill] donet
[cycle.enrich_thin] donet
[cycle.skillopt] donet
[cycle.embed] start[dry-run] Would embed 0 chunks (0 stale found)
[cycle.embed] done
[orphans.scan] donett
[cycle.orphans] done
[cycle.schema_suggest] donet
[cycle.purge] donet
Dream cycle (partial) in 0.3s:
  ! lint        307 issue(s) found (dry-run, no writes)
  ✓ backlinks   1167 missing back-link(s) found (audit-only; run gbrain check-backlinks fix to materialize)
  ✓ sync        0 page(s) would sync, 0 would

&emsp;&emsp;现在真跑。**关键：要带 `--dir` 指向你的源目录**，否则 dream 的文件系统阶段（lint、backlinks、sync、synthesize）会被静默跳过，只剩 DB 侧阶段：

In [98]:
# dream 真跑：--dir 指向源目录，文件系统阶段才会运行
!gbrain dream --dir datasets/vc-brain-corpus

[cycle.lint] donet
[backlinks.scan] donett
[cycle.backlinks] done
[cycle.sync] start[gbrain phase] sync.resolve_repo
[gbrain phase] sync.load_active_pack
[gbrain phase] sync.detect_head
[cycle.sync] done
[cycle.synthesize] donet
[cycle.extract] donet
[cycle.extract_facts] donet
[cycle.resolve_symbol_edges] donet
[cycle.patterns] donet
[cycle.recompute_emotional_weight] donet
[cycle.consolidate] donet
[cycle.propose_takes] donet
[cycle.grade_takes] donet
[cycle.calibration_profile] donet
[cycle.conversation_facts_backfill] donet
[cycle.enrich_thin] donet
[cycle.skillopt] donet
[cycle.embed] startEmbedded 0 chunks (0 stale found)
[cycle.embed] done
[orphans.scan] donett
[cycle.orphans] done
[cycle.schema_suggest] donet
[cycle.purge] donet
Dream cycle (partial) in 0.4s:
  ! lint        0 fix(es) applied, 307 remaining
  ✓ backlinks   1167 missing back-link(s) found (audit-only; run gbrain check-backlinks fix to materialize)
  ✓ sync        +0 added, ~0 modified, -0 deleted
  - synthesize 

&emsp;&emsp;真跑输出是一份**逐阶段的体检报告**（`Dream cycle (partial)`），我们挑关键几行读：

- `! lint  0 fix(es) applied, 307 remaining`——扫出 307 个格式问题（缺 `created` / `title` 字段等），但**一个都没自动改**。

- `OK backlinks  1167 missing back-link(s) found (audit-only)`——发现 1167 条缺失的反向链接，**只报告、不补**（要补得另跑 `gbrain check-backlinks fix`）。

- `! orphans  138 orphan page(s) out of 155 total`——揪出 138 个孤儿页（没有任何链接连进/连出的页）；这 155 含你在 5.3 节「日常闭环」刚加的那页。

- `OK schema-suggest  5 suggestions emitted`——还顺手提了 5 条 schema 改进建议。

- `OK sync  +0 added, ~0 modified, -0 deleted`——sync 靠 git diff 做增量检测，这份语料正好在一个 git 仓库内、没有未提交变更，所以跑通了、报 0 变更（源目录若不是 git 仓库，这一阶段会如实报 `Not a git repository`）。

- 其余 `consolidate / propose_takes / grade_takes` 都是 0——没有矛盾、没有待提炼的事实，无活可干。

&emsp;&emsp;看出来了吗？**dream 几乎什么都没"修"**。它扫出 307 个格式问题、1167 条缺失反链、138 个孤儿页，然后……就停在那里把报告交给你。这不是它没用，恰恰是它的**设计哲学：审计优先（audit-first）——dream 负责发现漂移，但默认不静默改你的大脑**。这正好回答了那个最常见的疑问："dream 到底修了哪些文件？"——**它给的是一份体检报告，不是一张自动修理单**。修不修、怎么修，是你的决定。

&emsp;&emsp;那"修"怎么修？是单独、显式的动作。比如 dream 报的格式问题，用 `lint --fix` 试着自动修——但这里有个诚实的细节要你亲眼看到：

In [99]:
# lint --fix：尝试自动修复格式问题（注意看末尾"auto-fixed"是几个）
!gbrain lint datasets/vc-brain-corpus --fix

[lint.pages] start
calendar/calendar-2026-03-15-0000-founder-check-in.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-16-0030-board-meeting.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-16-0033-gp-meeting.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-17-0025-due-diligence-call.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-17-0026-technical-review.md:
  L1 missing-title: Frontmatter missing required field: title
  L1 missing-created: Frontmatter missing required field: created

calendar/calendar-2026-03-17-0031-hiring-interview.md:
  L1 missing-

&emsp;&emsp;真跑你会看到 `307 issue(s) in 154 page(s)`，但末尾是 **`0 auto-fixed`**——`missing-created` 这类问题 `--fix` **修不了**，得你手动给 frontmatter 补上 `created` 字段。这再次印证了那条边界：**工具帮你发现问题，但很多修复仍是你的手动决定**。

> **【踩坑预警 · dream 的文件系统阶段必须带 `--dir`】**　不带 `--dir`，dream 的 lint / backlinks / sync / synthesize 这些扫源文件的阶段会被**静默跳过**（输出里会看到 "pass --dir to run filesystem phases"），你以为 dream 全跑了，其实只跑了 DB 侧那半。要让它真正扫你的 Markdown 源、查格式和反链，必须 `gbrain dream --dir <你的源目录>`。另外 sync 阶段还要求源目录是 **git 仓库**，否则那一个阶段会如实报 `Not a git repository` 失败（其它阶段照常）。

> **【踩坑预警 · dream 是"体检仪"不是"自动修理工"】**　别指望 `gbrain dream` 跑完你的大脑就自动变干净了——它默认 **audit-only**：lint `0 applied`、backlinks `audit-only`，它**报告** 漂移但**不静默改** 你的大脑。真要修是显式动作（`lint --fix` 修可修的、`check-backlinks fix` 补反链、`delete` 软删旧页），而且不是每条都能自动修（`missing-created` 实测就是 `0 auto-fixed`）。这种"只发现、不擅自改"恰恰是它对你大脑负责的方式——下一节 6.2 节「防 KB 漂移」我们会把这条"维护是你的责任"的边界讲透。

> **【踩坑预警 · 梦循环的自动后台执行靠常驻 worker，而 PGLite 起不了】**　梦循环可以手动 `gbrain dream` 跑一次（PGLite 上没问题，它是一次性内联执行）；但"每晚自动后台执行"靠的是常驻 worker daemon（`gbrain jobs work`）——**这个常驻 worker 在 PGLite 上跑不起来**。要分清原因：**不是"队列机制不支持 PGLite"**（`jobs submit` / `jobs stats` / `--dry-run` 在 PGLite 上都正常），真正卡住的是 PGLite 的**独占文件锁**（单进程单连接），一个常驻 worker 会和你的主进程抢锁——这正是第二节第 8.4 节讲过的那条边界。**PGLite 上想真正执行任务有两条路**：① 一次性任务用 `gbrain jobs submit <name> --follow`（当前进程内联跑完）；② 想要"夜间自动维护"的效果，就用系统 cron/launchd 定时触发 `gbrain dream`（第 7.3 节讲这个管线）。

&emsp;&emsp;我们可以用 `jobs stats` 看看作业队列的状态，确认队列健康：

In [100]:
# jobs stats：作业队列健康面板（等待/活跃/卡住的作业数）
!gbrain jobs stats

Job Stats (last 24h):
  No jobs in the last 24 hours.

  Queue health: 0 waiting, 0 active, 0 stalled
  Lease pressure (1h): 0 bounces


&emsp;&emsp;在我们的个人大脑上，`jobs stats` 会显示"No jobs in the last 24 hours"和"Queue health: 0 waiting, 0 active"——空队列是健康的常态。你也可以用 `jobs submit sync --dry-run` 试着提交一个作业看看（dry-run 不会真执行）：

In [101]:
# jobs submit --dry-run：演示作业提交（不真执行，只看会提交什么）
!gbrain jobs submit sync --dry-run

[DRY RUN] Would submit job:
  Name: sync
  Queue: default
  Priority: 0
  Max attempts: 3
  Data: {}


### 6.2 防 KB 漂移

&emsp;&emsp;运维命令是工具，但真正要建立的是**意识**：你的大脑会漂移，而且漂移往往是**不可检测的**——这是第一节五章讲过的最重要的诚实边界。我们把它落到两个具体场景：

&emsp;&emsp;**场景一：旧页被当成新事实引用**。你三个月前写了一页"我们的定价是 X"，后来定价改了但那页没更新。`think` 综合时还会引用那页，给你一个过时的答案——而且它不知道自己过时了。**这就是 KB 自我投毒**：脏数据混在干净数据里，污染了综合结果。

&emsp;&emsp;**场景二：多页自相矛盾**。页 A 说"项目优先 AI 安全"，页 B 说"项目优先增长"，两页都没错（可能是不同时间的决策），但放在一起就矛盾了。`think` 的冲突检测能发现这类矛盾（还记得第 4 章 Gaps 段里的"冲突与矛盾"吗），但前提是你得定期跑它、定期看。

> **【诚实边界 · GBrain 不承诺 set-and-forget】**　不要把 GBrain 当成"建好就不用管"的系统。任何长期运行的知识库都会漂移，GBrain 也不例外。**它的诚实在于：它不假装自己永远正确，而是给你工具（`doctor`、`dream`、`think` 的冲突检测）去发现漂移**。但工具不会自己跑——你得建立运维节奏。这一点上 GBrain 和市面上号称"全自动记忆"的方案不同，它把"维护是你的责任"这件事明明白白告诉你。

&emsp;&emsp;落到可执行的运维节奏，给你一份可以直接抄的清单：

> **【运维巡检清单 · 可直接抄】**
> - **周频**：`gbrain lint <source-dir>`——保证新导入的源是干净的。
> - **月频**：`gbrain doctor`——大脑全身体检，处理 FAIL 级问题。
> - **事件触发**：发现信息过期/矛盾时，立刻 `gbrain delete <slug>` 软删旧页，或更新它。
> - **可选自动化**：cron/launchd 定时跑 `gbrain dream`（PGLite 替代 worker daemon，见第 7.3 节）。

### 6.3 版本化记忆：软删 vs 硬删

&emsp;&emsp;维护大脑免不了要删东西——删过期的页、删错误的事实。但删之前你得知道一件事：**GBrain 的删除是分级的，删错了能不能救回来，取决于你用哪种删法**。这一节我们把软删和硬删讲清楚，真跑一遍。

&emsp;&emsp;GBrain 的 `delete` 命令默认是**软删**——它把页标记为删除、从搜索和列表里隐藏，但**72 小时内可以恢复**。我们真跑一下，删掉你在 5.3 节「日常闭环」刚加的 `decisions/q3-ai-safety-fund` 页（拿个新加的演示页练手，不动原始语料）：

In [102]:
# delete：默认软删——页被隐藏但 72h 内可恢复（recoverable）
!gbrain delete decisions/q3-ai-safety-fund

{
  "status": "soft_deleted",
  "slug": "decisions/q3-ai-safety-fund",
  "recoverable_until": "now + 72h via restore_page"
}


&emsp;&emsp;真跑返回 `"recoverable_until": "now + 72h via restore_page"`。注意这是**软删**——页还在数据库里，只是被标记隐藏了。我们用 `list` 确认它确实从列表消失了：

In [103]:
# list：确认软删后的页不再出现在列表里（但数据还在，可恢复）
!gbrain list

calendar/calendar-2026-03-15-0000-founder-check-in	calendar-event	2026-06-26	Calendar 2026 03 15 0000 Founder Check In
calendar/calendar-2026-03-16-0030-board-meeting	calendar-event	2026-06-26	Calendar 2026 03 16 0030 Board Meeting
calendar/calendar-2026-03-16-0033-gp-meeting	calendar-event	2026-06-26	Calendar 2026 03 16 0033 Gp Meeting
calendar/calendar-2026-03-17-0025-due-diligence-call	calendar-event	2026-06-26	Calendar 2026 03 17 0025 Due Diligence Call
calendar/calendar-2026-03-17-0026-technical-review	calendar-event	2026-06-26	Calendar 2026 03 17 0026 Technical Review
calendar/calendar-2026-03-17-0031-hiring-interview	calendar-event	2026-06-26	Calendar 2026 03 17 0031 Hiring Interview
calendar/calendar-2026-03-18-0002-1-on-1	calendar-event	2026-06-26	Calendar 2026 03 18 0002 1 On 1
calendar/calendar-2026-03-18-0027-founder-check-in	calendar-event	2026-06-26	Calendar 2026 03 18 0027 Founder Check In
calendar/calendar-2026-03-18-0028-lp-update-call	calendar-event	2026-06-26	Calenda

&emsp;&emsp;`q3-ai-safety-fund` 这一页从 `list` 里消失了。但它没有被物理删除——这就是软删的安全网。如果你删错了，72 小时内可以用 `restore` 救回来：

In [104]:
# restore：恢复软删的页（72h 内有效），页重新出现在搜索和列表里
!gbrain restore decisions/q3-ai-safety-fund

# 确认恢复成功
!gbrain list | grep q3-ai-safety-fund

{
  "status": "restored",
  "slug": "decisions/q3-ai-safety-fund"
}
decisions/q3-ai-safety-fund	decision	2026-06-26	Q3 AI 安全专项基金决议


&emsp;&emsp;`restore` 返回 `"status": "restored"`，`q3-ai-safety-fund` 页又回到了列表里。这就是版本化记忆的价值——**删除是可逆的**。下面这张是软删 vs 硬删语义辨析的截图：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171249906.png" width=50%></div>

&emsp;&emsp;那硬删呢？软删过了 72 小时恢复窗口后，autopilot 的 purge 阶段会把它物理删除（hard-delete），那时就真的找不回了。所以这里有一条铁律：

> **【删除铁律 · 先软删，给自己留后悔药】**　删任何页之前，记住 GBrain 的删除是"软删先行"——`gbrain delete` 默认软删，72h 内可 `restore`。**正确做法**：发现一页该删时，先软删，观察几天确认它真的不需要了，再让恢复窗口自然过期变成硬删。**在没有确认页面真的不再需要前，不要跳过这个恢复窗口**——MCP 工具里的 `forget_fact` 也是同样的软删语义（它管的是"事实级"软删，`delete` 管的是"页级"）。在你确信之前，让删除可逆，这是对自己大脑负责的态度。

### 6.4 运维判读练习

&emsp;&emsp;最后一个小练习。`doctor` 跑出来一堆 OK/WARN/FAIL，新手最容易犯的错是"看到 WARN 就慌"或者"看到 FAIL 也不管"。我们做个判读练习——下面三种 `doctor` 输出，你判断哪些要立刻处理、哪些可以先放着：

> **【动手练习 · doctor 输出分类判读】**
> 1. `[WARN] retrieval_reflex_health → policy skill not installed`——**该不该管？** 这是个"功能没装"的提示，不影响核心检索。**判读：可以先放着**，等你需要那个 skill 时再装。
> 2. `[FAIL] resolver_health → 34 error(s)`——**该不该管？** resolver 是大脑解析页面的核心，FAIL 级。**判读：要管**，跑 `gbrain doctor --fix` 看能否自动修，不行就深查。
> 3. `[WARN] skill_brain_first → 1 skill does external lookups without brain-first signal`——**该不该管？** 这关系到 brain-first 契约的合规。**判读：建议管**，它在提醒你某个 skill 没遵守"答前先查"，可能造成失忆。

&emsp;&emsp;判读的核心原则很简单：**FAIL 级看核心功能（resolver/pgvector/embedding）必须处理；WARN 级分两种——影响 brain-first 合规的要管，纯"功能没装"的可以按需**。养成这个判读习惯，你就不会被 `doctor` 的输出吓到，也不会漏掉真正的问题。

&emsp;&emsp;本章我们把大脑从"建好"推进到"长期可用"：`lint/doctor/dream` 三件套自检（`dream` 真跑出的是一份体检报告，发现漂移但默认不替你改——修复是你的决定）、KB 漂移的运维意识、软删 vs 硬删的安全网。**维护这条线的痛点也回收了**——大脑会漂移，但你有了发现和修复漂移的完整手段。到这里，一个个人大脑的完整生命周期（设计→建库→查询→接入→维护）就走完了。下一章，我们抬头看看更大的图景：当你从个人走向团队、从单一 RAG 走向混合架构、从手动维护走向自动化时，GBrain 还能怎么用。

---

## <center>第 7 章：进阶延伸（三条路径）</center>

&emsp;&emsp;前六章我们搭完了一个完整的个人大脑。但 GBrain 的能力边界远不止于此。这一章我们抬头看三条进阶路径：从个人走向**团队**（多人知识隔离）、从单一编译式 RAG 走向**混合架构**（编译 wiki + 向量 RAG 并用）、从手动维护走向**自动化**（cron 增量管线）。

&emsp;&emsp;先说清这一章的定位——**这是速览，不是完整实操**。每条路径只讲"是什么 + 什么时候用 + 关键效果"，每条 10-15 分钟，让你知道它存在、知道什么场景该考虑它。三条路径都有对应的部署指南供你后续深入。其中**团队 brain 的"内容面"我们会真跑一次演示**（因为它和个人大脑同机制、能跑通）；至于多 source 安全隔离、两套 RAG 混合架构、cron 调度这些需要额外环境配置的部分，我们用真实截图展示、命令片段只看不跑——在这里真跑会因配置问题卡住流程。

### 7.1 团队 brain：内容与安全

&emsp;&emsp;**什么时候用**：当大脑不再只服务一个人，而是要给一个团队用——团队的文档、决策、人员、策略需要一个统一的、能问的知识库，而且不同人能看到的知识还不一样（Alice 能看客户 A 的资料，Bob 不能），这时候就需要团队 brain。

&emsp;&emsp;这一节我们破例真跑一遍——因为团队 brain 的"内容面"和你前面建的个人大脑用的是**同一套机制**（schema 分类 + wikilink 自动建图 + think 综合），完全可以真跑；只有后面的多 source 安全隔离需要额外配置，那部分我们才用截图展示。先看**一个真实的团队知识库长什么样**。课程附带了一份真实的团队文档语料 `datasets/team-docflow-corpus/`，它是一个 SaaS 团队（虚构产品 DocFlow）的完整知识库，60 个中文文件，跨四个域：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>team-docflow-corpus：一个团队知识库</font></p>
<div class="center">

| 源目录 | 文件数 | 内容 |
|--------|--------|------|
| `policies/` | 14 | 运维策略（安全、SLA、备份、灾备、发布） |
| `architecture/` | 16 | 架构决策（认证、网关、存储、多租户、监控） |
| `api-specs/` | 15 | API 规格（Auth、Document、Billing、Webhook 等） |
| `team/` | 15 | 人员档案 + 团队页（谁负责哪个模块） |

</div>

&emsp;&emsp;导入它。这里有个**真实小坑** 要注意，还正好接上第 2 章的自定义类型那一段：`extract links` 有两种抽取来源，行为不一样。`--source db`（从数据库里已编译的内容抽）带一个**目录前缀白名单**——它只认 gbrain 内置的实体目录（`people` / `companies` / `meetings` / `concepts` 等），不在白名单里的目录前缀会被直接过滤掉。而这份团队语料用的是 `team/` / `policies/` / `architecture/` / `api-specs/` 这些**自定义目录**，全都不在白名单里，所以 `--source db` 抽出来是 **0 条边**。解法是改用 `--source fs`（直接从原始文件抽），它**没有这个白名单限制**，按实际页 slug 匹配，就能把 234 条边都建出来：

In [107]:
# 同样不带 --force：全局 embedding_disabled 已清、且是全新库（理由同 §3.7 建 brain-vc）
!gbrain init --path ./output/brain-team --embedding-model dashscope:text-embedding-v3
!gbrain import datasets/team-docflow-corpus
# 注意：自定义目录（team/policies/...）不在 --source db 的白名单里，要用 --source fs
!gbrain extract links --source fs --dir datasets/team-docflow-corpus

Setting up local brain with PGLite (no server needed)...
  Embedding: dashscope:text-embedding-v3 (1024d)
  Expansion: openai:gpt-5.2
[init] Using schema pack: gbrain-base-v2 (override with --schema-pack <name>)

Brain ready at ./output/brain-team
60 pages. Engine: PGLite (local Postgres).

Existing brain detected. To wire up the v0.10.3 knowledge graph:
  gbrain extract links --source db        (typed link backfill)
  gbrain extract timeline --source db     (structured timeline backfill)
  gbrain stats                            (verify links > 0)

When you outgrow local: gbrain migrate --to supabase

--- GBrain Mod Status ---
Skills: 51 loaded
GStack: not found
  Install GStack for coding skills:
  git clone https://github.com/garrytan/gstack.git ~/.claude/skills/gstack
  cd ~/.claude/skills/gstack && ./setup
Resolver: skills/RESOLVER.md
Soul audit: run `gbrain soul-audit` to customize agent identity
Retrieval reflex: on by default (entity pointers injected per turn)
  Install the po

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171255350.png" width=50%></div>

&emsp;&emsp;60 页团队文档、234 条边，跨 architecture / api-spec / policy / person / team 五类。真正见效的是 `think`——一个团队成员问一句，它就跨整个团队知识库综合出答案。比如新人入职，问一句"DocFlow 的认证是怎么设计的？谁负责？涉及哪些安全策略？"：

In [106]:
!gbrain think "DocFlow 的认证是怎么设计的？谁负责？涉及哪些安全策略？" --model deepseek:deepseek-v4-flash

# DocFlow 的认证是怎么设计的？谁负责？涉及哪些安全策略？

# DocFlow 认证设计、责任团队及安全策略

## 认证设计
DocFlow 采用 **JWT（JSON Web Token）** 作为无状态认证机制，核心设计包括：
- **Access Token**：15 分钟有效期，包含 `user_id`、`org_id`、`roles` 等声明 [architecture/auth-design]。
- **Refresh Token**：30 天有效期，单次使用，刷新后旧 token 即时失效 [architecture/auth-design]。
- **签名算法**：RS256（非对称），公钥可分发给所有子服务独立验签，支持水平扩展 [architecture/auth-design]。
- **鉴权流程**：API Gateway 验 JWT → 提取用户/组织/角色 → 下游服务（如 doc-service）进行文档级 ACL 检查（RBAC + 文档级权限混合模型）[architecture/permission-model][architecture/api-gateway-design]。
- **Token 轮换策略**：Access Token 每次刷新自动轮换；Refresh Token 设备绑定、密码修改或可疑登录时全部失效；服务账号 mTLS 证书 90 天有效期，cert-manager 自动轮换 [policies/token-rotation-policy]。

## 负责团队与个人
- **平台工程团队（Platform Engineering）**：负责认证系统与权限系统的开发与运维，成员 Bob Morgan 是认证和权限模型的主要开发者 [team/team-platform-engineering][team/bob-morgan]。
- **安全团队（Security）**：负责安全策略制定、渗透测试与合规审计，安全工程师 Lead Rachel Sun 主导安全策略制定 [team/team-security][team/rachel-sun]。
- **工程总监 Alice Chen**：对认证架构（JWT vs Session）进行最终技术决策 [team/alice-chen]。

## 涉及的

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171256140.png" width=50%></div>

&emsp;&emsp;一个问题，think 跨 **40 页** 文档把三件事拼到了一起：**认证怎么设计**（JWT / RS256，来自 `architecture/auth-design`）、**谁负责**（Bob Morgan 开发、Alice Chen 拍板，来自 `team/` 档案）、**涉及哪些策略**（`security-policy` / `network-security-policy`，来自 `policies/`），每条都带可追溯的来源页。这就是团队知识库的核心价值——**新人不用追着老人一个个问，问大脑就行，答案还能溯源**。

&emsp;&emsp;但团队 brain 比个人 brain 多一道坎——**安全隔离**。团队 brain 的核心设计，是**多 source 切分**：不是把所有人的知识堆进一个大脑，而是按 source 物理隔离成多个"脑中之脑"。典型的三 source 架构是 `shared`（全员共享）、`customers`（客户资料，按权限可见）、`internal`（内部敏感）。关键问题来了：怎么保证 Alice 查询时绝对看不到 Bob 的 source？（下面这部分需要多 source 配置，我们用截图展示。）

&emsp;&emsp;答案是这一节最重要的认知——**隔离不能靠 LLM 自觉，必须在数据库层强制**。GBrain 用 `federated_read` 的 SQL scope 做隔离：查询在 SQL 层就被限定在授权的 source 范围内，LLM 根本拿不到越权数据。下面这张是零泄漏隔离的实测——同一套查询，Alice 和 Bob 看到完全不同的结果：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171252615.png" width=50%></div>

> **【核心认知 · 隔离靠 SQL 不靠 LLM】**　很多人做团队知识隔离的第一反应是"在 prompt 里告诉 LLM 别泄露别人的数据"——这是错的。**LLM 不理解 source 边界，prompt 约束随时可能被绕过**（一句巧妙的提问就可能套出越权数据）。GBrain 的做法是在 `federated_read` 的 SQL scope 层强制隔离：越权数据在数据库查询阶段就被过滤掉，根本不进入 LLM 的上下文。**这是数据库层的硬隔离，不是 LLM 层的软约束**——这才是团队知识安全的底线。

&emsp;&emsp;隔离效果可以观察、可以验证。下面这张是 scoped think 的输出——同一个问题，不同身份的人跑，因为可见的 source 不同，`think` 综合出来的答案也不同：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171253148.png" width=50%></div>

&emsp;&emsp;关于从个人走向团队的升级路径，一句话概括：**PGLite → Postgres + OAuth**。个人大脑用的 PGLite 是单机嵌入式的，团队大脑需要迁移到 Postgres（支持多连接、行级权限 RLS），再配上 OAuth 做身份认证。迁移命令是 `gbrain migrate --to supabase`，但完整的多 source 配置 + OAuth 设置在配套的《团队大脑部署指南》里，这里不展开。

> **【诚实边界 · 团队 brain 能做但要自己搭】**　团队 brain 不是开箱即用的——它需要你自己搭 Postgres、配 source 切分、设 OAuth。也只有团队 brain（Postgres 引擎）才有真正的 worker daemon（`gbrain jobs work`）做后台梦循环，这点和第 6 章呼应。**它是 GBrain 的能力，但是一条需要工程投入的路径**。先把个人大脑用熟，确认团队真的有多人隔离需求时，再上 Postgres。完整搭建见配套的《团队大脑部署指南》（`团队大脑部署指南.md`）。

### 7.2 编译 wiki + 向量 RAG 混合

&emsp;&emsp;**什么时候用**：当你既有"稳定的、值得编译成大脑的知识"（决策、人物、长期背景），又有"实时变动的、不值得编译的文档"（每天更新的文档、临时资料）——这时候单一的 GBrain 不够，就需要混合架构。

&emsp;&emsp;这里要先说清编译式 RAG（GBrain）和传统向量 RAG 的关系：**两者不是竞争关系，是互补关系**。GBrain 擅长稳定知识的预编译综合，传统向量 RAG 擅长实时文档的即时检索。混合架构就是让两者各管各的擅长场景，用一个**路由器** 根据 query 类型分流：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171243977.png" width=50%></div>

&emsp;&emsp;这张双源接入图展示了混合架构的形态——同一个查询入口，背后接了 GBrain 和向量 RAG 两个源。路由是关键，不是"两个都用就更好"。混合架构有两个容易混淆的概念需要分清：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>混合架构选型对照</font></p>
<div class="center">

| 方案 | 适用场景 | 复杂度 | 维护成本 |
|------|---------|--------|---------|
| GBrain only | 知识以稳定/半静态为主 | 低 | 低 |
| 向量 RAG only | 文档实时变动、不需图谱综合 | 低 | 中 |
| 混合架构 | 稳定知识 + 实时文档都要 | 高（多一层路由） | 高 |

</div>

> **【常见误区 · 架构级 RRF ≠ GBrain 内部 RRF】**　第二节我们讲过 GBrain 内部用 RRF 融合向量和 BM25 两路检索——那是**检索层** 的融合。混合架构里也有 RRF，但那是**系统级** 的融合：把 GBrain 和向量 RAG 两个独立系统的结果融合成一个排序。**这是两个不同层次**：内部 RRF 在一个系统里融合两路打分，架构级 RRF 在两个系统之间融合结果。别把它们搞混——它们解决的是不同粒度的问题。

&emsp;&emsp;路由做得好不好，是可以测量的，不靠感觉。下面这张是路由评估的真实输出，量化了路由器把不同 query 分流到正确系统的精度：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171241746.png" width=50%></div>

> **【诚实边界 · 混合架构引入路由复杂度】**　混合架构不是"免费的更好"——它引入了路由这一层新的复杂度，而**路由错误比单源更难调试**（query 被分到错的系统，结果不对，你还得先判断是检索错了还是路由错了）。**建议**：先用 GBrain only 跑通你的场景，确认单源真的不够用（确实有大量实时文档需要即时检索）时，再上混合架构。别一开始就追求"全都要"。完整实现见配套的《混合架构实现指南》（`混合架构实现指南.md`）。

### 7.3 cron 自动化编译管线

&emsp;&emsp;**什么时候用**：当你的知识源在持续变动（比如一个 git 仓库，你每天往里 commit 新文档），你不想每次手动 import，而是希望大脑自动跟着源更新——这就需要自动化编译管线。

&emsp;&emsp;这条路径最吸引人的，是一个性能数字：**增量 sync 比全量 sync 快一个数量级**。原因很直观——全量 sync 每次重新处理所有文件，增量 sync 只处理自上次以来变更的文件。我们实测了这个对比：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171237670.png" width=50%></div>

&emsp;&emsp;这张对比图里，增量 sync 只处理变更的几个文件、秒级完成，而全量 sync 要重跑整个语料、慢一个数量级（在那份语料、那台机器上单次实测约一分钟量级 vs 1 秒级；这是单次小样本测量、不是基准数字，换机器/换语料量级会变，重点看"差一个数量级"这个量级关系而非具体秒数）。这个差异在知识库越大时越显著——大语料全量 sync 可能要几分钟，增量只要几秒。

&emsp;&emsp;增量 sync 的实现逻辑是一个三步循环：**git 变更检测 → 增量 import → 持续 self-wiring**。每次你往 git 仓库 commit，管线检测到 diff、只 import 变更的文件、然后增量更新图谱。下面这张是 cron 管线脚本的真实样貌：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171252481.png" width=50%></div>

&emsp;&emsp;在 macOS 上，比 cron 更稳定的调度方式是 launchd。关键命令是用一个 plist 配置注册一个定时任务（比如每 15 分钟触发一次 `gbrain sync`），我们展示配置思路但不真跑：

```bash
# launchd 定时 sync 的核心思路（仅展示，不真跑——需要配 plist 文件）
# gbrain 自带 sync --install-cron 帮你装持续 sync 守护
# !gbrain sync --install-cron
# 或手动配 launchd plist，每 15 分钟跑：cd <repo> && gbrain sync --repo .
```

> **【踩坑预警 · macOS 上别用 crontab、改用 launchd；关键字段是 StartInterval】**　为什么上面说"launchd 比 cron 稳"？这背后是个真实教训：第一次用 crontab 装这条管线，在非交互环境下直接被 `EXIT:143（SIGTERM）` 杀掉——macOS 那个 setuid root 的 crontab 在**非 TTY 环境** 会被系统干掉。换成 **launchd LaunchAgent**（用户级、不需要 root、加载也不需要 TTY）就正常了：`launchctl load <plist>` 返回 `EXIT:0`、`launchctl list` 能看到任务。plist 里最关键的字段是 **`StartInterval`**——它是触发间隔的**秒数**，演示用 `120`（每 2 分钟）方便观察，**生产环境改成你的周期**（每 15 分钟就写 `900`）。再补一条同源铁律（和第 2 章那条 APFS 呼应）：**被调度的源仓库也必须放在本机 APFS 盘（`~/` 下）、不能放外接 exFAT 盘**——macOS 的 LaunchAgent 对可移动磁盘有沙箱限制，写入会报 `Operation not permitted`，调度跑了但全部写失败。完整七环节部署见配套的《自动化增量管线部署指南》。

> **【踩坑预警 · 增量 sync 依赖 git commit 锚点】**　增量 sync 用 git commit 作为"变更检测"的锚点——**未 commit 的文件不会被 sync 到大脑**。这是设计如此，不是 bug：管线靠 git diff 知道哪些文件变了。**正确做法**：要让某个文件进入增量管线，先 `git commit` 它。**另外再次提醒**：PGLite 上 `gbrain jobs work`（worker daemon）不可用，所以"自动后台编译"在个人大脑上就是靠这套 cron/launchd 触发的 sync 来实现的——这正好串起第 6 章那个"PGLite 用 cron 替代 worker daemon"的边界。完整部署见配套的《自动化增量管线部署指南》（`自动化增量管线部署指南.md`）。

&emsp;&emsp;本章我们速览了三条进阶路径：团队 brain 的 SQL 硬隔离、混合架构的两种 RAG 互补、自动化 cron 管线的增量加速。它们的共同点是——**都是从"个人玩得转"走向"规模化用得起"的桥**。你现在不需要立刻搭它们，但你知道了它们存在、知道了什么场景该考虑它们、也知道了去哪查完整实操。这就是速览的价值：给你一张更大的地图。

---

## <center>第 8 章：能力自测 + 选型判断</center>

&emsp;&emsp;三节课走完了。从第一节的原理范式、第二节的核心功能，到这一节的端到端落地——我们一起把 GBrain 从一个概念，变成了一个能跑、能用、能接进 Agent 的真实工具。这最后一章我们不讲新东西，而是回头盘点：你现在到底能做什么，以及一个比"会用"更重要的能力——**知道什么时候该用、什么时候不该用**。

### 8.1 能力锚点：你现在能做什么

&emsp;&emsp;先一起做一次诚实的能力盘点。这节课我们围绕三条链路展开，分别对应三个优先级（P0 最核心、P1 主体、P2 进阶）。下面我们把它们摆成一张表，你对照着逐条问自己"这个我真的能独立做出来吗"：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三条能力链路盘点</font></p>
<div class="center">

| 链路 | 你现在能做什么 | 对应章节 | 验收信号 |
|------|--------------|---------|---------|
| P0 · Agent 长记忆 | MCP 接入 + 写 brain-first 契约 + 跨会话验证 | 第 5 章 | 新会话零历史命中上个会话的事实 |
| P1 · 个人第二大脑 | 设计 schema + 建带 embedding 的库 + 导入多源 + think 综合 | 第 2-4 章 | think 给出带引用 + 空白分析的答案 |
| P2 · 团队/进阶入门 | 知道团队隔离/混合架构/自动化三条路径的入口 | 第 7 章 | 能判断什么场景该考虑哪条进阶路径 |

</div>

&emsp;&emsp;如果这三条你都能打勾——P0 你能让 Agent 跨会话记忆、P1 你能从零搭个人大脑、P2 你知道进阶往哪走——那这节课的目标就达成了。特别是 P0 和 P1 这两条核心链路，是你必须真正掌握的；P2 是"知道存在、需要时能查"的程度就够了。

### 8.2 选型判断框架：什么时候该用 GBrain

&emsp;&emsp;会用一个工具，和知道什么时候该用它，是两回事。后者更难，也更值钱——因为**选错工具比不用更贵**。这一节我们把第一节讲过的选型决策树补全，凑成一个完整的判断框架。

&emsp;&emsp;核心的选型逻辑是问自己几个问题，顺着决策树走：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171237728.png" width=50%></div>

&emsp;&emsp;我们把这棵树用文字说清楚，顺着三问走一遍：**第一问，要的是对话记忆还是知识综合？** 如果是"记住用户在对话里说的每句话、随时增删改"——那是高频对话记忆，选 Mem0/Zep，别用 GBrain。如果是"把一堆文档/决策/人物编译成一个能跨源综合查询的大脑"——那才是 GBrain 的主场。**第二问，知识稳定吗？** GBrain 擅长稳定/半静态知识；如果还混有大量实时变动的文档，那就考虑混合架构。**第三问，需要多人隔离吗？** 需要就上团队 brain（Postgres + SQL scope）。

&emsp;&emsp;选型的另一半，是诚实地知道 GBrain 的边界。下面我们用一张速查表把它的适用边界讲清楚：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>GBrain 诚实边界速查</font></p>
<div class="center">

| 边界维度 | 说明 |
|---------|------|
| 知识形态 | 适合静态/半静态知识预编译，不为高频对话记忆的频繁增删改设计 |
| 规模适用性 | 个人/小团队规模适用得很好；超大语料（远超个人量级）需要评估引擎选型（PGLite→Postgres）和查询延迟，建议先小规模验证 |
| KB 漂移 | 不承诺 set-and-forget——需要定期 lint/doctor/dream 维护（第 6 章） |
| 维护责任 | 它给你发现漂移的工具，但维护节奏要你自己建立 |

</div>

> **【选型诚实提示 · 规模问题要实测，不要套数字】**　关于"GBrain 能撑多大的语料"，这里要诚实说一句：**没有一个适用于所有场景的固定上限**。个人和小团队规模的知识库，GBrain 跑得很好；当语料规模显著增长时，查询延迟和引擎选型（PGLite 的单机限制 vs Postgres 的扩展能力）会成为需要评估的因素。**正确做法**：不要去记某个"多少万词/多少篇"的数字当成硬上限——那种数字往往没有官方支撑，而且实际能撑多大高度依赖你的硬件、语料结构和查询模式。要判断 GBrain 能不能扛住你的规模，最可靠的办法是**用你自己的真实语料小规模实测**：导入一部分，看 import 速度、查询延迟、`think` 响应时间，再决定要不要上 Postgres。规模是个实测问题，不是查表问题。

### 8.3 能力自测题 A：选型判断

&emsp;&emsp;光看框架不够，我们做两道题检验你的判断力。第一道是选型判断题——给你三个真实场景，你判断该用 GBrain / Mem0 / 向量 RAG / 混合架构里的哪个。先自己想，再看参考答案。

> **【自测题 A · 三个场景，你选哪个？】**
> 1. **场景一**：你想做一个客服机器人，要记住每个用户在对话中提到的偏好（喜欢什么、讨厌什么），随时更新。
> 2. **场景二**：你想把团队过去两年的所有会议纪要、决策记录、人物档案编译成一个大脑，让 Agent 跨会话帮你做会前准备。
> 3. **场景三**：你的知识库里既有稳定的产品手册（适合编译），又有每天更新的客户工单（实时变动），两类都要能查。

&emsp;&emsp;下面我们对一遍参考答案和判断依据：

&emsp;&emsp;**场景一选 Mem0/Zep**。关键词是"对话中""随时更新""偏好"——这是典型的高频对话记忆，要的是实时增删改，正好是 Mem0/Zep 的主场。拿 GBrain 做这个，"写入时编译"的机制会拖慢响应。

&emsp;&emsp;**场景二选 GBrain**。关键词是"会议纪要/决策/人物档案"（结构化、半静态的知识）"跨会话会前准备"——这正是 GBrain 的核心场景，编译 + 图谱 + think 综合全用得上。如果团队需要多人隔离，再升级到团队 brain（Postgres）。

&emsp;&emsp;**场景三选混合架构**。关键词是"稳定的产品手册 + 每天更新的工单"——稳定知识交给 GBrain 编译，实时工单交给向量 RAG，用路由器分流。这正是第 7.2 节讲的混合架构适用场景。

&emsp;&emsp;我们要注意，这三道题的答案不是"背出来的"，而是"判断出来的"——核心判断依据始终是那两个问题：**是对话记忆还是知识综合？知识是稳定的还是实时变动的？** 抓住这两个问题，大部分选型你都能自己判断。

### 8.4 能力自测题 B：为你的领域设计大脑（开放题）

&emsp;&emsp;最后一道是开放设计题，没有标准答案——它检验的是你能不能把这节课学的东西，迁移到你自己的真实场景。这也是我们对第 2 章 schema 设计的一次综合巩固。

> **【自测题 B · 设计你自己领域的大脑】**　为你自己的工作领域，设计一套大脑 schema。具体要求写出：
> - **3 个 type**：你这个领域要记录哪三类核心知识？（可以从 `gbrain-base-v2` 的 15 个类型里选，也可以自定义）
> - **2 个关系动词（link verb）**：这些 type 之间用什么关系连接？
> - **画一张草图**：用 wikilink 把它们连起来，画出一个最小的知识图谱。

&emsp;&emsp;给你一个填好的示范，对照着做你自己的。假设我是做开源项目维护的：

&emsp;&emsp;**我的 3 个 type**：`issue`（问题）、`pr`（合并请求）、`contributor`（贡献者）。**我的 2 个关系动词**：`fixes`（PR 修复某 issue）、`authored_by`（PR 由某 contributor 提交）。**草图**：一个 `pr` 页通过 `[[fixes::issue-123]]` 连到 issue，通过 `[[authored_by::alice]]` 连到 contributor——这样 `think` 就能跨这三类页综合出"alice 修了哪些 issue、有哪些 PR 还没合"这种全局视角的答案。

&emsp;&emsp;把你自己领域的这套草图写出来——这一步做完，你就真正把 GBrain 从"跟着步骤敲命令"内化成了"能用它解决我自己的问题"。这才是整套方法最核心的迁移——从「跟着步骤敲命令」到「能用它解决自己的问题」。

### 8.5 本节小结

&emsp;&emsp;三节课，我们一起走完了 GBrain 的完整旅程：第一节理解它的范式（编译式 RAG、self-wiring、诚实边界），第二节拆开它的引擎（存储、检索、综合、生态），这一节把所有零件组装成一条端到端的链路，亲手搭出了一个能跨会话记忆的第二大脑。下面这张全景图，就是你走完本节后真正搭出来的东西——从散落知识，到编译大脑，到 `think` 带引用 + 盲区的跨源综合，再到 MCP 接入 Agent 的跨会话长记忆，一条完整闭环：

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260626171237708.png" width=82%></div>

&emsp;&emsp;回到这节课开头钉在墙上的三个痛点，我们逐一回收了：**知识散落各处**——第 3 章 import 进多源知识，编译成一个有图谱的大脑；**搜索只返回页面**——第 4 章 think 直接给你成文答案 + 空白分析；**Agent 每开新会话就失忆**——第 5 章 brain-first 契约 + 跨会话验证，让记忆活在大脑里而非会话里。三个痛点，三条链路，都闭环了。

&emsp;&emsp;如果回头对照开篇列的 7 件产物，你会发现它们都已落到这三条链路里：schema 设计、两个真实库（`brain-l3` 与 `brain-vc`）、多源知识导入流程、一次 `think` 会前综合的真实输出、写进 `CLAUDE.md` 的 brain-first 契约、一次跨会话长记忆的受控验证，以及上一节 §8.2 补全的 GBrain vs Mem0/Zep 选型框架——七件，你都带走了。

&emsp;&emsp;但比这些具体技能更重要的，是我们一路反复在强调的两个判断力：**一个是诚实**——GBrain 不假装自己永远正确，它用 Gaps 标盲区、用 doctor 查漂移、用软删留后悔药，这种"知道自己不知道什么"的诚实，是我们用任何知识工具都该保持的态度。**另一个是选型**——知道 GBrain 的主场是稳定知识的跨源综合，知道它不是高频对话记忆的方案，知道规模问题要实测而非查表。

&emsp;&emsp;现在，把这把"尺"用到你自己的地方去——你工作中有哪些散落的知识值得编译成大脑？你的哪个 Agent 正受困于"每次从零开始"？你下一个项目，是该上 GBrain、还是 Mem0、还是混合架构？这些问题的答案，不在这节课里，在你自己的场景里。